# Librerie utilizzate

In [ ]:
import numpy as np
import pandas as pd
import math
from scipy.stats import t
import matplotlib.pyplot as plt
from collections import deque
import math, heapq
import os, csv
from collections.abc import Mapping

import matplotlib.pyplot as plt
from typing import Mapping, Iterable, Any, Sequence


# Librerie utilizzate per le distribuzione e il genratore di numeri casuali

In [2]:
#-------------------------------------------------------------------------- 
 #This is an ANSI C library for generating random variates from six discrete 
 #distributions
 #
 #     Generator         Range (x)     Mean         Variance
 #
 #     Bernoulli(p)      x = 0,1       p            p*(1-p)
 #     Binomial(n, p)    x = 0,...,n   n*p          n*p*(1-p)
 #     Equilikely(a, b)  x = a,...,b   (a+b)/2      ((b-a+1)*(b-a+1)-1)/12
 #     Geometric(p)      x = 0,...     p/(1-p)      p/((1-p)*(1-p))
 #     Pascal(n, p)      x = 0,...     n*p/(1-p)    n*p/((1-p)*(1-p))
 #     Poisson(m)        x = 0,...     m            m
 #
 #and seven continuous distributions
 #
 #     Uniform(a, b)     a < x < b     (a + b)/2    (b - a)*(b - a)/12 
 #     Exponential(m)    x > 0         m            m*m
 #     Erlang(n, b)      x > 0         n*b          n*b*b
 #     Normal(m, s)      all x         m            s*s
 #     Lognormal(a, b)   x > 0            see below
 #     Chisquare(n)      x > 0         n            2*n 
 #     Student(n)        all x         0  (n > 1)   n/(n - 2)   (n > 2)
 #
 #For the a Lognormal(a, b) random variable, the mean and variance are
 #
 #                       mean = exp(a + 0.5*b*b)
 #                   variance = (exp(b*b) - 1) #exp(2*a + b*b)
 #
 #Name              : rvgs.c  (Random Variate GeneratorS)
 #Author            : Steve Park & Dave Geyer
 #Language          : ANSI C
 #Latest Revision   : 10-28-98
 #Translated by     : Philip Steele 
 #Language          : Python 3.3
 #Latest Revision   : 3/26/14
 # 
 #--------------------------------------------------------------------------


from math import log,sqrt,exp
# -------------------------------------------------------------------------
# This is an ANSI C library for multi-stream random number generation.  
#  * The use of this library is recommended as a replacement for the ANSI C 
#  * rand() and srand() functions, particularly in simulation applications 
#  * where the statistical 'goodness' of the random number generator is 
#  * important.  The library supplies 256 streams of random numbers; use 
#  * SelectStream(s) to switch between streams indexed s = 0,1,...,255.
#  *
#  * The streams must be initialized.  The recommended way to do this is by
#  * using the function PlantSeeds(x) with the value of x used to initialize 
#  * the default stream and all other streams initialized automatically with
#  * values dependent on the value of x.  The following convention is used 
#  * to initialize the default stream:
#  *    if x > 0 then x is the state
#  *    if x < 0 then the state is obtained from the system clock
#  *    if x = 0 then the state is to be supplied interactively.
#  *
#  * The generator used in this library is a so-called 'Lehmer random number
#  * generator' which returns a pseudo-random number uniformly distributed
#  * 0.0 and 1.0.  The period is (m - 1) where m = 2,147,483,647 and the
#  * smallest and largest possible values are (1 / m) and 1 - (1 / m)
#  * respectively.  For more details see:
#  * 
#  *       "Random Number Generators: Good Ones Are Hard To Find"
#  *                   Steve Park and Keith Miller
#  *              Communications of the ACM, October 1988
#  *
#  * Name            : rngs.c  (Random Number Generation - Multiple Streams)
#  * Authors         : Steve Park & Dave Geyer
#  * Language        : ANSI C
#  * Latest Revision : 09-22-98
#  * Translated by     : Philip Steele 
#  * Language          : Python 3.3
#  * Latest Revision   : 3/26/14
#  *
#  * ------------------------------------------------------------------------- 

from time import time

#global consts
MODULUS = 2147483647 #/* DON'T CHANGE THIS VALUE                  */
MULTIPLIER = 48271      #/* DON'T CHANGE THIS VALUE                  */
CHECK = 399268537  #/* DON'T CHANGE THIS VALUE                  */
STREAMS = 256        #/* # of streams, DON'T CHANGE THIS VALUE    */
A256 = 22925      #/* jump multiplier, DON'T CHANGE THIS VALUE */
DEFAULT = 123456789  #/* initial seed, use 0 < DEFAULT < MODULUS  */

#statics
stream = 0
initialized = 0
seed = [DEFAULT]
for i in range(1,STREAMS):
  seed.append(DEFAULT)


def random(): 
  #/* ---------------------------------------------------------------------
  #* Random is a Lehmer generator that returns a pseudo-random real number
  #* uniformly distributed between 0.0 and 1.0.  The period is (m - 1)
  #* where m = 2,147,483,647 amd the smallest and largest possible values
  #* are (1 / m) and 1 - (1 / m) respectively.
  #* ---------------------------------------------------------------------
  #*/
  global seed

  Q = int(MODULUS / MULTIPLIER)
  R = int(MODULUS % MULTIPLIER)

  t = int(MULTIPLIER * (seed[stream] % Q) - R * int(seed[stream] / Q))
  if (t > 0):
    seed[stream] = int(t)
  else:
    seed[stream] = int(t + MODULUS)

  return float(seed[stream] / MODULUS)

def plantSeeds(x): 
  # /* --------------------------------------------------------------------
  #  * Use this function to set the state of all the random number generator
  #  * streams by "planting" a sequence of states (seeds), one per stream,
  #  * with all states dictated by the state of the default stream.
  #  * The sequence of planted states is separated one from the next by
  #  * 8,367,782 calls to Random().
  #  * ---------------------------------------------------------------------
  #  */
  global initialized
  global stream
  global seed

  Q = int(MODULUS / A256)
  R = int(MODULUS % A256)

  initialized = 1
  s = stream                             #/* remember the current stream */
  selectStream(0)                        #/* change to stream 0          */
  putSeed(x)                             #/* set seed[0]                 */
  stream = s                             #/* reset the current stream    */
  for j in range(1,STREAMS):
    x = int(A256 * (seed[j - 1] % Q) - R * int((seed[j - 1] / Q)))
    if (x > 0):
      seed[j] = x
    else:
      seed[j] = x + MODULUS
  

def putSeed(x):
  # /* -------------------------------------------------------------------
  #  * Use this (optional) procedure to initialize or reset the state of
  #  * the random number generator according to the following conventions:
  #  *    if x > 0 then x is the initial seed (unless too large)
  #  *    if x < 0 then the initial seed is obtained from the system clock
  #  *    if x = 0 then the initial seed is to be supplied interactively
  #  * --------------------------------------------------------------------
  #  */
  global seed 

  ok = False

  if (x > 0):
    x = x % MODULUS  
                            # correct if x is too large  
  if (x < 0): 
    x = time()
    x = x % MODULUS
  
  if (x == 0):
    while (ok == False): 
      line = input("\nEnter a positive integer seed (9 digits or less) >> ")
      x = int(line)
      ok = (0 < x) and (x < MODULUS)
      if (ok == False):
        print("\nInput out of range ... try again\n")
    
    
  seed[stream] = int(x)


def getSeed():
  # /* --------------------------------------------------------------------
  #  * Use this (optional) procedure to get the current state of the random
  #  * number generator.
  #  * --------------------------------------------------------------------
  #  */
  global seed
  return seed[stream]


def selectStream(index):
  #/* ------------------------------------------------------------------
  #* Use this function to set the current random number generator
  #* stream -- that stream from which the next random number will come.
  #* ------------------------------------------------------------------
  #*/
  global stream 

  stream = index % STREAMS
  if (initialized == 0) and (stream != 0):   #/* protect against        */
    plantSeeds(DEFAULT)                     #/* un-initialized streams */
    


  
def testRandom():
  # /* -------------------------------------------------------------------
  #  * Use this (optional) procedure to test for a correct implementation.
  #  * -------------------------------------------------------------------
  #  */

  ok = False

  selectStream(0)                  #/* select the default stream */
  putSeed(1)                       #/* and set the state to 1    */
  for i in range(0,10000):
    u = random()
  x = getSeed()                    #/* get the new state value   */
  ok = (x == CHECK)                #/* and check for correctness */
  
  selectStream(1)                  #/* select stream 1                 */
  plantSeeds(1)                    #/* set the state of all streams    */
  x = getSeed()                    #/* get the state of stream 1       */
  ok = (ok==True) and (x == A256)           #/* x should be the jump multiplier */
  if (ok==True):
    print("\n The implementation of Rngs.py is correct")
  else:
    print("\n ERROR - the implementation of Rngs.py is not correct")

  

def Bernoulli(p):
  #========================================================
  #Returns 1 with probability p or 0 with probability 1 - p. 
  #NOTE: use 0.0 < p < 1.0                                   
  #========================================================
  
  if (random() < 1 - p):
    return(0)
  else:
    return(1)


def Binomial(n,p):
  #================================================================ 
  #Returns a binomial distributed integer between 0 and n inclusive. 
  #NOTE: use n > 0 and 0.0 < p < 1.0
  #================================================================
  
  x = 0

  for i in range(0,n):
    x += Bernoulli(p)
  return (x)

def Equilikely(a,b):
  #===================================================================
  #Returns an equilikely distributed integer between a and b inclusive. 
  #NOTE: use a < b
  #===================================================================
  return (a + int((b - a + 1) * random()))

def Geometric(p):
  #====================================================
  #Returns a geometric distributed non-negative integer.
  #NOTE: use 0.0 < p < 1.0
  #====================================================
  #

  return (int(log(1.0 - random()) / log(p)))


def Pascal(n,p):
  #================================================= 
  #Returns a Pascal distributed non-negative integer. 
  #NOTE: use n > 0 and 0.0 < p < 1.0
  #=================================================
  #
   
  x = 0

  for i in range(0,n):
    x += Geometric(p)
  return (x)

def Poisson(m):
  #================================================== 
  #Returns a Poisson distributed non-negative integer. 
  #NOTE: use m > 0
  #==================================================
  # 
  t = 0.0
  x = 0

  while (t < m): 
    t += Exponential(1.0)
    x += 1
  
  return (x - 1)

def Uniform(a,b):
  #=========================================================== 
  #Returns a uniformly distributed real number between a and b. 
  #NOTE: use a < b
  #===========================================================
  #
  return (a + (b - a) * random())

def Exponential(m):
  #=========================================================
  #Returns an exponentially distributed positive real number. 
  #NOTE: use m > 0.0
  #=========================================================
  #
  return (-m * log(1.0 - random()))

def Hyperexponential(p, l1, l2):
  # ====================================================================
  # Returns a hyperexponential distributed positive real number.
  # NOTE: 0.0 < p < 1.0, l1 > 0.0, l2 > 0.0
  # It returns an exponential with rate l1 with probability p,
  # otherwise with rate l2 with probability (1 - p)
  # ====================================================================
  #
  if (random() < p):
    return Exponential(1.0 / l1)
  else:
    return Exponential(1.0 / l2)

def Erlang(n,b):
  #================================================== 
  #Returns an Erlang distributed positive real number.
  #NOTE: use n > 0 and b > 0.0
  #==================================================
  #
  x = 0.0

  for i in range(0,n): 
    x += Exponential(b)
  return (x)

def Normal(m,s):
  #========================================================================
  #Returns a normal (Gaussian) distributed real number.
  #NOTE: use s > 0.0
  #
  #Uses a very accurate approximation of the normal idf due to Odeh & Evans, 
  #J. Applied Statistics, 1974, vol 23, pp 96-97.
  #========================================================================
  #
  p0 = 0.322232431088     
  q0 = 0.099348462606
  p1 = 1.0                
  q1 = 0.588581570495
  p2 = 0.342242088547     
  q2 = 0.531103462366
  p3 = 0.204231210245e-1  
  q3 = 0.103537752850
  p4 = 0.453642210148e-4  
  q4 = 0.385607006340e-2

  u = random()
  if (u < 0.5):
    t = sqrt(-2.0 * log(u))
  else:
    t = sqrt(-2.0 * log(1.0 - u))

  p   = p0 + t * (p1 + t * (p2 + t * (p3 + t * p4)))
  q   = q0 + t * (q1 + t * (q2 + t * (q3 + t * q4)))

  if (u < 0.5):
    z = (p / q) - t
  else:
    z = t - (p / q)

  return (m + s * z)

def Lognormal(a,b):
  # ==================================================== 
  #Returns a lognormal distributed positive real number. 
  #NOTE: use b > 0.0
  #====================================================
  #
  return (exp(a + b * Normal(0.0, 1.0)))

def Chisquare(n):
  #=====================================================
  #Returns a chi-square distributed positive real number. 
  #NOTE: use n > 0
  #=====================================================
  #
  x = 0.0

  for i in range(0,n):
    z  = Normal(0.0, 1.0)
    x += z * z

  return (x)


def Student(n):
  #=========================================== 
  #Returns a student-t distributed real number.
  #NOTE: use n > 0
  #===========================================
  #
  return (Normal(0.0, 1.0) / sqrt(Chisquare(n) / n))

def testFunctions():
  #tests to ensure that all variates match what was produced by C version of program (with the same order and parameters)

  #bernoulli 
  bern = []
  for i in range(0,10):
    bern.append(Bernoulli(.65))

  if (bern == [0,0,1,0,1,0,1,1,1,1]):
    print("Bernoulli test passed")
  else:
    print("FIX BERNOULLI!")

  #binomial
  bino = Binomial(50,.19)
  if (bino == 7):
    print("Binomial test passed")
  else:
    print("FIX BINOMIAL")
    print(bino)

  #equilikely 
  equi = Equilikely(32,108)
  if (equi ==77):
    print("Equilikely test passed")
  else:
    print("FIX EQUILIKELY")

  #geo test
  geo = []
  for i in range(0,10):
    geo.append(Geometric(.93))

  if (geo == [0,8,4,21,7,3,17,5,14,1]):
    print("Geo test passed")
  else:
    print("Fix geo!")
    print(geo)

  #pascal test
  pas = Pascal(87,.93)
  if (pas == 1407):
    print("Pascal test passed!")
  else:
    print("FIX PASCAL")

  #poisson test
  pois = Poisson(13.3)
  if (pois == 15):
    print("Poisson test passed!")
  else:
    print("FIX POISSON!")

  #uniform test
  uni = Uniform(32,108)
  if (round(uni,6) == 78.976127):
    print("Uniform test passed!")
  else:
    print("FIX UNIFORM. Produced: ", uni)
    print("Expected: 78.976127")

  #exp test
  exp = Exponential(4.7)
  if (round(exp,6) == 4.796404):
    print("Exponential test passed!")
  else:
    print("FIX EXP. Produced: ", exp)
    print("expected: 4.796404")

  #erlang test
  erl = Erlang(41,.08)
  if (round(erl,6) == 2.824873):
    print("Erland test passed!")
  else:
    print("FIX ERLANG Produced: ", erl)
    print("Expected: 2.824873")

  #normal test
  norm = Normal(8.9,4)
  if (round(norm,6) == 8.374243):
    print("Normal test passed!")
  else:
    print("FIX NORMAL Produced: ",norm)
    print("Expected: 8.374243")

  #lognormal test
  lnorm = Lognormal(1.31,1.6)
  if (round(lnorm,6) == 5.533064):
    print("Lognormal test passed!")
  else:
    print("FIX LOGNORMAL Produced: ", lnorm)
    print("Expected: 5.533064")

  #chisq test
  chisq = Chisquare(39)
  if (round(chisq,6) == 33.634524):
    print("Chisquare test passed!")
  else:
    print("FIX CHI-SQUARE - Produced: ", chisq)
    print("Expected: 33.634524")

  #t test
  stu = Student(61)
  if (round(stu,6) == -1.429058):
    print("Student test passed!")
  else:
    print("FIX STUDENT - Produced: ", stu)
    print("Expected: -1.429058")

# Codice sviluppo simulazione modello base e anche con lambda variabile

In [3]:
# STREAM IDs
STREAM_ARR = 0
STREAM_A_1   = 1
STREAM_A_2   = 2
STREAM_A_3   = 3
STREAM_B   = 4
STREAM_P   = 5

In [4]:

def simulate_ps_ABAPA_fast(
    lmb, mu_A1, mu_B, mu_A2, mu_P, mu_A3,
    threshold_time, eps=1e-12
):
   

    def stage_server_name(pos): 
        return ["A","B","A","P","A"][pos-1]

    def a_visit_index_after(pos): 
        return [1,1,2,2,3][pos-1]
    
  

    def draw_demand_for(pos):
        if pos == 1:
            selectStream(STREAM_A_1)   
            return Exponential(1/mu_A1)
        if pos == 2: 
            selectStream(STREAM_B)   
            return Exponential(1/mu_B)
        if pos == 3: 
            selectStream(STREAM_A_2)   
            return Exponential(1/mu_A2)
        if pos == 4: 
            selectStream(STREAM_P)   
            return Exponential(1/mu_P)
        if pos==5: 
            selectStream(STREAM_A_3)   
            return Exponential(1/mu_A3)

    # Stato server con PS: S = service tag, heap di finish tag F
    class PSServer:
        __slots__ = ("name","c","S","n","heap","N_time_accum")
        def __init__(self,name,c=1.0):
            self.name=name; self.c=c
            self.S=0.0; self.n=0
            self.heap=[]   # (F, jid, segobj)
            self.N_time_accum=0.0

    servers = {k:PSServer(k,1.0) for k in ("A","B","P")}

    # Stato globale
    selectStream(STREAM_ARR) 
    t=0.0
    mean_inter=1.0/float(lmb)
    next_arrival = t + Exponential(mean_inter)
    job_id=0
    jobs_rows=[]
    jobs_info = {}  # jid -> dict

    # Helper: avanzamento tempo delta aggiornando S e N_time_accum
    def advance(delta):
        nonlocal t
        if delta <= 0: 
            return
        # accumulo N(t)*dt
        for srv in servers.values():
            srv.N_time_accum += srv.n * delta
        # avanço dei service tag
        for srv in servers.values():
            if srv.n>0:
                srv.S += (srv.c / srv.n) * delta
        t += delta

    # Ammetti segmento (visita) su un server PS
    def admit_segment(jid,pos,server_name,arrival_t):
        srv = servers[server_name]
        d = draw_demand_for(pos)

        # numero di job attivi all'ammissione (incluso questo)
        n_at_admit = srv.n + 1

        seg = {
            "job_id": jid, 
            "pos": pos, 
            "server": server_name,
            "a_visit": a_visit_index_after(pos),
            "arrival_stage": arrival_t,
            "IN": arrival_t,                    # in PS puro ammissione = arrivo
            "S_admit": srv.S,                   # tag all'ammissione
            "Demand": d,
            "N_at_Admit": n_at_admit
        }
        F = seg["S_admit"] + d
        heapq.heappush(srv.heap, (F, jid, seg))
        srv.n += 1
        return seg

    # Main loop
    INF = math.inf
    def arrivals_open(): 
        return (next_arrival is not None and next_arrival <= threshold_time)

    def work_left():
        if arrivals_open(): 
            return True
        return any(srv.n>0 for srv in servers.values())

    while work_left():
        # Prossimo completamento su ciascun server in tempo reale
        next_comp_time = INF
        comp_srv = None

        for srv in servers.values():
            if srv.n==0 or not srv.heap: 
                continue
            F_min, _, _ = srv.heap[0]
            if F_min <= srv.S + 1e-15:
                dt_srv = 0.0
            else:
                dS = F_min - srv.S
                dt_srv = dS * (srv.n / srv.c)
            tc = t + dt_srv
            if tc < next_comp_time:
                next_comp_time = tc
                comp_srv = srv

        ta = next_arrival if arrivals_open() else INF
        t_next = min(ta, next_comp_time)
        if math.isinf(t_next):
            break

        # Avanza il tempo fino al prossimo evento
        advance(t_next - t)

        # Evento
        if abs(t_next - ta) <= 1e-12:
            # ARRIVO
            selectStream(STREAM_ARR)
            job_id += 1
            jid = job_id
            jobs_info[jid] = {"arrival0": t, "stage_recs": [], "demand_total": 0.0}
            admit_segment(jid, 1, stage_server_name(1), t)

            ia = Exponential(mean_inter)
            cand = t + ia
            next_arrival = cand if cand <= threshold_time else None

        else:
            # COMPLETION su comp_srv
            F_min, jid, seg = heapq.heappop(comp_srv.heap)
            comp_srv.n -= 1

            IN = seg["IN"]
            OUT = t
            Demand = seg["Demand"]
            Wait = 0.0
            Service = OUT - IN  # tempo effettivo trascorso nella visita (PS)

            # Valori "istantanei" all'ammissione
            n_at_admit = seg["N_at_Admit"]
            avgN_during = n_at_admit  # tua scelta: riportiamo lo stesso valore

            jobs_info[jid]["stage_recs"].append({
                "pos": seg["pos"], 
                "server": seg["server"], 
                "id_label": f"{jid}[{seg['a_visit']}]",
                "IN": IN, "OUT": OUT, "Demand": Demand, "Wait": Wait,
                "Service": Service, 
                "N_at_Admit": n_at_admit, 
                "AvgN_during": avgN_during
            })
            jobs_info[jid]["demand_total"] += Demand

            # Prosegue o chiude
            if seg["pos"] < 5:
                admit_segment(jid, seg["pos"]+1, stage_server_name(seg["pos"]+1), t)
            else:
                # chiude job: aggrega le visite
                recs = sorted(jobs_info[jid]["stage_recs"], key=lambda r: r["pos"])
                row = {
                    "OrigID": jid,
                    "Arrival0": jobs_info[jid]["arrival0"],
                    "Completion": t,
                    "TempoRispostaTotal": t - jobs_info[jid]["arrival0"],           # sojourn time
                    "TempoServizioTotaleSeIlJObFosseSOlo": jobs_info[jid]["demand_total"]  # somma dei Demand_i
                }
                for i, rec in enumerate(recs, 1):
                    row.update({
                        f"Server_{i}": rec["server"],
                        f"ID_{i}": rec["id_label"],
                        f"IN_{i}": rec["IN"],
                        f"OUT_{i}": rec["OUT"],
                        f"Demand_{i}": rec["Demand"],
                        f"Wait_{i}": rec["Wait"],
                        # rinominato: Response_TIme_{i} = Service effettivo (OUT-IN)
                        f"Response_TIme_{i}": rec["Service"],
                        f"N_at_Admit_{i}": rec["N_at_Admit"],
                        f"AvgN_during_{i}": rec["AvgN_during"],
                    })
                jobs_rows.append(row)

    # DF finale con i nuovi nomi
    K=5
    base_cols=[
        "OrigID","Arrival0","Completion",
        "TempoRispostaTotal","TempoServizioTotaleSeIlJObFosseSOlo"
    ]
    per_stage=[]
    for i in range(1,K+1):
        per_stage += [
            f"Server_{i}", f"ID_{i}", f"IN_{i}", f"OUT_{i}",
            f"Demand_{i}", f"Wait_{i}", f"Response_TIme_{i}",
            f"N_at_Admit_{i}", f"AvgN_during_{i}"
        ]
    df = pd.DataFrame(jobs_rows)
    for c in base_cols + per_stage:
        if c not in df.columns: 
            df[c]=pd.NA
    df = df[base_cols+per_stage].sort_values("OrigID").reset_index(drop=True)

    # AvgN globali
    t_end = t if t>0 else 1.0
    stats = {
        "AvgN_A": servers["A"].N_time_accum / t_end,
        "AvgN_B": servers["B"].N_time_accum / t_end,
        "AvgN_P": servers["P"].N_time_accum / t_end,
        "T_end": t_end
    }
    return df, stats


#----------------------------------------

# Fasce orarie giornaliere
fasce = ["00-06", "06-09", "09-12", "12-17", "17-21", "21-24"]

# Giorni della settimana
giorni = ["Sabato", "Domenica"]

def calculate_lmb(t, lambda_settimanale):
    """
    Calcola il tasso di arrivo lambda in base al tempo t (in secondi)
    e al profilo settimanale fornito.
    """
    # Calcola il giorno della settimana e l'ora del giorno
    giorno_idx = int(t // 86400) % 7  # 86400 secondi in un giorno
    ora_del_giorno = (t % 86400) / 3600  # Converti in ore

    giorno = giorni[giorno_idx]

    # Determina la fascia oraria
    if 0 <= ora_del_giorno < 6:
        fascia = "00-06"
    elif 6 <= ora_del_giorno < 9:
        fascia = "06-09"
    elif 9 <= ora_del_giorno < 12:
        fascia = "09-12"
    elif 12 <= ora_del_giorno < 17:
        fascia = "12-17"
    elif 17 <= ora_del_giorno < 21:
        fascia = "17-21"
    else:
        fascia = "21-24"

    print(lambda_settimanale[giorno][fascia])

    return lambda_settimanale[giorno][fascia]

def simulate_ps_ABAPA_mod(
    lmb, mu_A1, mu_B, mu_A2, mu_P, mu_A3,
    threshold_time,settimana
):
   

    def stage_server_name(pos): 
        return ["A","B","A","P","A"][pos-1]

    def a_visit_index_after(pos): 
        return [1,1,2,2,3][pos-1]
    
  

    def draw_demand_for(pos):
        if pos == 1:
            selectStream(STREAM_A_1)   
            return Exponential(1/mu_A1)
        if pos == 2: 
            selectStream(STREAM_B)   
            return Exponential(1/mu_B)
        if pos == 3: 
            selectStream(STREAM_A_2)   
            return Exponential(1/mu_A2)
        if pos == 4: 
            selectStream(STREAM_P)   
            return Exponential(1/mu_P)
        if pos==5: 
            selectStream(STREAM_A_3)   
            return Exponential(1/mu_A3)

    # Stato server con PS: S = service tag, heap di finish tag F
    class PSServer:
        __slots__ = ("name","c","S","n","heap","N_time_accum")
        def __init__(self,name,c=1.0):
            self.name=name; self.c=c
            self.S=0.0; self.n=0
            self.heap=[]   # (F, jid, segobj)
            self.N_time_accum=0.0

    servers = {k:PSServer(k,1.0) for k in ("A","B","P")}

    # Stato globale
    selectStream(STREAM_ARR) 
    t=0.0
    lmb = calculate_lmb(t,settimana)
    mean_inter=1.0/float(lmb)
    next_arrival = t + Exponential(mean_inter)
    job_id=0
    jobs_rows=[]
    jobs_info = {}  # jid -> dict

    # Helper: avanzamento tempo delta aggiornando S e N_time_accum
    def advance(delta):
        nonlocal t
        if delta <= 0: 
            return
        # accumulo N(t)*dt
        for srv in servers.values():
            srv.N_time_accum += srv.n * delta
        # avanço dei service tag
        for srv in servers.values():
            if srv.n>0:
                srv.S += (srv.c / srv.n) * delta
        t += delta

    # Ammetti segmento (visita) su un server PS
    def admit_segment(jid,pos,server_name,arrival_t):
        srv = servers[server_name]
        d = draw_demand_for(pos)

        # numero di job attivi all'ammissione (incluso questo)
        n_at_admit = srv.n + 1

        seg = {
            "job_id": jid, 
            "pos": pos, 
            "server": server_name,
            "a_visit": a_visit_index_after(pos),
            "arrival_stage": arrival_t,
            "IN": arrival_t,                    # in PS puro ammissione = arrivo
            "S_admit": srv.S,                   # tag all'ammissione
            "Demand": d,
            "N_at_Admit": n_at_admit
        }
        F = seg["S_admit"] + d
        heapq.heappush(srv.heap, (F, jid, seg))
        srv.n += 1
        return seg

    # Main loop
    INF = math.inf
    def arrivals_open(): 
        return (next_arrival is not None and next_arrival <= threshold_time)

    def work_left():
        if arrivals_open(): 
            return True
        return any(srv.n>0 for srv in servers.values())

    while work_left():
        # Prossimo completamento su ciascun server in tempo reale
        next_comp_time = INF
        comp_srv = None

        for srv in servers.values():
            if srv.n==0 or not srv.heap: 
                continue
            F_min, _, _ = srv.heap[0]
            if F_min <= srv.S + 1e-15:
                dt_srv = 0.0
            else:
                dS = F_min - srv.S
                dt_srv = dS * (srv.n / srv.c)
            tc = t + dt_srv
            if tc < next_comp_time:
                next_comp_time = tc
                comp_srv = srv

        ta = next_arrival if arrivals_open() else INF
        t_next = min(ta, next_comp_time)
        if math.isinf(t_next):
            break

        # Avanza il tempo fino al prossimo evento
        advance(t_next - t)

        # Evento
        if abs(t_next - ta) <= 1e-12:
            # ARRIVO
            selectStream(STREAM_ARR)
            job_id += 1
            jid = job_id
            jobs_info[jid] = {"arrival0": t, "stage_recs": [], "demand_total": 0.0}
            admit_segment(jid, 1, stage_server_name(1), t)

            ia = Exponential(mean_inter)
            cand = t + ia
            next_arrival = cand if cand <= threshold_time else None

        else:
            # COMPLETION su comp_srv
            F_min, jid, seg = heapq.heappop(comp_srv.heap)
            comp_srv.n -= 1

            IN = seg["IN"]
            OUT = t
            Demand = seg["Demand"]
            Wait = 0.0
            Service = OUT - IN  # tempo effettivo trascorso nella visita (PS)

            # Valori "istantanei" all'ammissione
            n_at_admit = seg["N_at_Admit"]
            avgN_during = n_at_admit  # tua scelta: riportiamo lo stesso valore

            jobs_info[jid]["stage_recs"].append({
                "pos": seg["pos"], 
                "server": seg["server"], 
                "id_label": f"{jid}[{seg['a_visit']}]",
                "IN": IN, "OUT": OUT, "Demand": Demand, "Wait": Wait,
                "Service": Service, 
                "N_at_Admit": n_at_admit, 
                "AvgN_during": avgN_during
            })
            jobs_info[jid]["demand_total"] += Demand

            # Prosegue o chiude
            if seg["pos"] < 5:
                admit_segment(jid, seg["pos"]+1, stage_server_name(seg["pos"]+1), t)
            else:
                # chiude job: aggrega le visite
                recs = sorted(jobs_info[jid]["stage_recs"], key=lambda r: r["pos"])
                row = {
                    "OrigID": jid,
                    "Arrival0": jobs_info[jid]["arrival0"],
                    "Completion": t,
                    "TempoRispostaTotal": t - jobs_info[jid]["arrival0"],           # sojourn time
                    "TempoServizioTotaleSeIlJObFosseSOlo": jobs_info[jid]["demand_total"]  # somma dei Demand_i
                }
                for i, rec in enumerate(recs, 1):
                    row.update({
                        f"Server_{i}": rec["server"],
                        f"ID_{i}": rec["id_label"],
                        f"IN_{i}": rec["IN"],
                        f"OUT_{i}": rec["OUT"],
                        f"Demand_{i}": rec["Demand"],
                        f"Wait_{i}": rec["Wait"],
                        # rinominato: Response_TIme_{i} = Service effettivo (OUT-IN)
                        f"Response_TIme_{i}": rec["Service"],
                        f"N_at_Admit_{i}": rec["N_at_Admit"],
                        f"AvgN_during_{i}": rec["AvgN_during"],
                    })
                jobs_rows.append(row)

    # DF finale con i nuovi nomi
    K=5
    base_cols=[
        "OrigID","Arrival0","Completion",
        "TempoRispostaTotal","TempoServizioTotaleSeIlJObFosseSOlo"
    ]
    per_stage=[]
    for i in range(1,K+1):
        per_stage += [
            f"Server_{i}", f"ID_{i}", f"IN_{i}", f"OUT_{i}",
            f"Demand_{i}", f"Wait_{i}", f"Response_TIme_{i}",
            f"N_at_Admit_{i}", f"AvgN_during_{i}"
        ]
    df = pd.DataFrame(jobs_rows)
    for c in base_cols + per_stage:
        if c not in df.columns: 
            df[c]=pd.NA
    df = df[base_cols+per_stage].sort_values("OrigID").reset_index(drop=True)

    # AvgN globali
    t_end = t if t>0 else 1.0
    stats = {
        "AvgN_A": servers["A"].N_time_accum / t_end,
        "AvgN_B": servers["B"].N_time_accum / t_end,
        "AvgN_P": servers["P"].N_time_accum / t_end,
        "T_end": t_end
    }
    return df, stats


#----------------------------------------




In [11]:

df_jobs, stats = simulate_ps_ABAPA_fast(
    lmb=1.2,
    mu_A1=5.0, mu_B=1.25, mu_A2=2.5, mu_P=2.5, mu_A3=10.0,
    threshold_time=100000,
  
)


print(stats)
print(df_jobs.head(2000).to_string(index=False))


In [5]:


# Ordine fisso delle colonne nel CSV
COLUMNS = ["ARR", "A1", "A2", "A3", "B", "P"]



def snapshot_streams():
    """Restituisce un NUOVO dict con i seed correnti per ciascuno stream."""
    out = {}
    for name, idx in [("ARR", STREAM_ARR),
                      ("A1", STREAM_A_1), ("A2", STREAM_A_2), ("A3", STREAM_A_3),
                      ("B", STREAM_B), ("P", STREAM_P)]:
        selectStream(idx)
        out[name] = getSeed()
    return out  # NUOVO dict, nessun globale riusato

def print_all_streams():
    """Stampa (debug) e ritorna una copia nuova dei seed."""
    d = snapshot_streams()
    for k, v in d.items():
        print(f"{k}: {v}")
    return d

def write_streams_csv(
    path: str,
    data: Mapping,                # {run_id: {ARR:..., A1:...}, ...}
    *,
    run_field: str = "RUN",
    columns: list[str] | None = None,
    encoding: str = "utf-8"
):
    # normalizza
    rows = [(run_id, inner) for run_id, inner in sorted(data.items())]
    if columns is None:
        columns = COLUMNS[:]  # usa ordine fisso definito sopra

    header = [run_field] + columns
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding=encoding, newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        for run_id, inner in rows:
            w.writerow([run_id] + [inner.get(c, "") for c in columns])



## Esecuzione simulazione

In [8]:

dfs_jobs = {}
dfs_seeds = {}

plantSeeds(12348948)

num_runs = 5
for i in range(num_runs):
    print(f"--- RUN {i} seeds PRIMA ---")
    _ = print_all_streams() 
    dfs_seeds[i] = snapshot_streams()    

    df_jobs, stats = simulate_ps_ABAPA_fast(
        lmb=1.2,
        mu_A1=5.0, mu_B=1.25, mu_A2=2.5, mu_P=2.5, mu_A3=10.0,
        threshold_time=400000,
    )
    dfs_jobs[i] = df_jobs

    print(f"--- RUN {i} seeds DOPO ---")
    _ = print_all_streams()               


write_streams_csv("PMCSN/seedNewTRans.csv", dfs_seeds, run_field="RUN", encoding="utf-8")


In [47]:
#Grafici per transitorio


def _normalize_items_with_seeds(
    dfs_runs,
    run_seeds=None
):
    if isinstance(dfs_runs, Mapping):
        raw = list(dfs_runs.items())
        items=[]
        for pos,(_,df) in enumerate(raw):
            seed = run_seeds[pos] if run_seeds and pos < len(run_seeds) else None
            label = str(seed) if seed is not None else f"run-{pos}"
            items.append((label,df))
        return items

    items=[]
    for i,df in enumerate(dfs_runs):
        seed = run_seeds[i] if run_seeds and i < len(run_seeds) else None
        label = str(seed) if seed is not None else f"run-{i}"
        items.append((label,df))
    return items



def plot_throughput_per_scenario(
    dfs_runs,
    *,
    run_seeds=None,
    x_col="Completion",
    bin_size=100.0,
    x_max=None,
    ema_alpha=None,
    per_unit="per_second",
    title="Throughput per scenario",
    xlabel="Tempo",
    ylabel=None,
    legend=True,
    grid=True,
    show=True
):
    items = _normalize_items_with_seeds(dfs_runs, run_seeds)
    if not items:
        raise ValueError("dfs_runs è vuoto.")

    if x_max is None:
        x_vals=[]
        for _,df in items:
            x_vals.append(pd.to_numeric(df[x_col],errors="coerce").max())
        x_max=float(pd.Series(x_vals).max())

    n_bins=int(np.ceil(x_max/bin_size))
    edges=np.linspace(0,n_bins*bin_size,n_bins+1)
    centers=(edges[:-1]+edges[1:])/2

    unit_factor=1.0
    unit_label="job/s"
    if per_unit=="per_minute":
        unit_factor=60; unit_label="job/min"
    elif per_unit=="per_hour":
        unit_factor=3600; unit_label="job/h"

    if ylabel is None:
        ylabel=f"Throughput ({unit_label})"

    fig,ax=plt.subplots()
    stats={}

    for label,df in items:
        x=pd.to_numeric(df[x_col],errors="coerce").dropna()
        x=x[(x>=0)&(x<=x_max)]
        counts,_=np.histogram(x,bins=edges)
        thr=(counts/bin_size)*unit_factor

        if ema_alpha and 0<ema_alpha<1:
            thr=pd.Series(thr).ewm(alpha=ema_alpha,adjust=False).mean().to_numpy()

        ax.plot(centers,thr,label=label)
        stats[label]=pd.DataFrame({"t":centers,"throughput":thr})

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_xlim(0,x_max)
    if grid: ax.grid(True,ls="--",alpha=0.5)
    if legend: ax.legend(title="Seed")

    if show: plt.show()
    return fig,ax,stats


#-------------------------------------------#


def plot_utilization_per_scenario(
    dfs_runs,
    *,
    run_seeds=None,
    servers=("A","B","P"),
    bin_size=100.0,
    x_max=None,
    ema_alpha=None,
    title="Utilizzazione per scenario e server",
    xlabel="Tempo",
    ylabel="Utilizzazione",
    legend=True,
    grid=True,
    show=True,
    y_min=0.0,
    y_max=1.0
):
    items = _normalize_items_with_seeds(dfs_runs, run_seeds)

    # trova K in modo robusto
    df0 = next((df for _, df in items if isinstance(df, pd.DataFrame) and not df.empty), None)
    if df0 is None:
        raise ValueError("Tutti i DataFrame sono vuoti.")
    server_cols = [c for c in df0.columns if c.startswith("Server_")]
    if not server_cols:
        raise ValueError("Non trovo colonne Server_1, Server_2, ...")
    K = max(int(c.split("_")[1]) for c in server_cols)

    # x_max
    if x_max is None:
        xmax = []
        for _, df in items:
            if not isinstance(df, pd.DataFrame) or df.empty:
                continue
            for c in df.columns:
                if c.startswith("OUT_"):
                    xmax.append(pd.to_numeric(df[c], errors="coerce").max())
        s = pd.Series(xmax).dropna()
        if s.empty:
            raise ValueError("Impossibile determinare x_max dai dati (OUT_).")
        x_max = float(s.max())

    # binning
    n_bins = int(np.ceil(x_max / bin_size))
    edges = np.linspace(0.0, n_bins * bin_size, n_bins + 1)
    centers = (edges[:-1] + edges[1:]) / 2.0

    fig, ax = plt.subplots()

    def add_interval(s, e, acc):
        s = max(0.0, min(float(s), x_max))
        e = max(0.0, min(float(e), x_max))
        if e <= s:
            return
        i = int(s / bin_size)
        j = int((e - 1e-12) / bin_size)
        i = max(i, 0)
        j = min(j, len(acc) - 1)
        for b in range(i, j + 1):
            bs, be = edges[b], edges[b + 1]
            acc[b] += max(0.0, min(e, be) - max(s, bs))

    # >>> per avere legenda SOLO seed:
    first_server = servers[0] if len(servers) > 0 else None

    for server in servers:
        for seed_label, df in items:
            if not isinstance(df, pd.DataFrame) or df.empty:
                continue

            busy = np.zeros(n_bins, dtype=float)

            for i in range(1, K + 1):
                sc = f"Server_{i}"
                ic = f"IN_{i}"
                oc = f"OUT_{i}"
                if sc not in df.columns or ic not in df.columns or oc not in df.columns:
                    continue

                mask = (df[sc] == server)
                if not mask.any():
                    continue

                ins = pd.to_numeric(df.loc[mask, ic], errors="coerce").to_numpy()
                outs = pd.to_numeric(df.loc[mask, oc], errors="coerce").to_numpy()

                events = []
                for a, b in zip(ins, outs):
                    if np.isnan(a) or np.isnan(b):
                        continue
                    if b <= a:
                        continue
                    events.append((a, +1))
                    events.append((b, -1))

                if not events:
                    continue

                events.sort()
                lvl = 0
                last = None
                for t, d in events:
                    if last is not None and lvl > 0:
                        add_interval(last, t, busy)
                    lvl += d
                    last = t

            util = np.clip(busy / bin_size, 0.0, 1.0)

            if ema_alpha is not None and 0 < ema_alpha < 1:
                util = pd.Series(util).ewm(alpha=ema_alpha, adjust=False).mean().to_numpy()

            # label SOLO per first_server → legenda = solo seed
            line_label = str(seed_label) if server == first_server else "_nolegend_"
            ax.plot(centers, util, label=line_label)

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_xlim(0, x_max)
    ax.set_ylim(y_min, y_max)
    if grid:
        ax.grid(True, ls="--", alpha=0.5)
    if legend:
        ax.legend(title="Seed")

    if show:
        plt.show()

    return fig, ax



#--------------------------------
def plot_rt_mean_per_scenario(
    dfs_runs,
    x_col="Completion",
    y_col="TempoRispostaTotal",
    *,
    run_seeds=None,
    bin_size=100.0,
    x_max=None,
    ema_alpha=None,
    # ---- controllo asse Y ----
    y_min=None,
    y_max=None,
    y_pad_frac=0.05,
    # ---- estetica ----
    title="RT medio (transitorio)",
    xlabel="Tempo (s)",
    ylabel="RT medio (s)",
    legend=True,
    grid=True,
    show=True
):
    items = _normalize_items_with_seeds(dfs_runs, run_seeds)
    if not items:
        raise ValueError("dfs_runs vuoto")

    # x_max
    if x_max is None:
        xmax = []
        for _, df in items:
            if isinstance(df, pd.DataFrame) and x_col in df.columns:
                xmax.append(pd.to_numeric(df[x_col], errors="coerce").max())
        s = pd.Series(xmax).dropna()
        if s.empty:
            raise ValueError(f"Impossibile determinare x_max: colonna {x_col!r} non valida.")
        x_max = float(s.max())

    # bin edges
    n_bins = int(np.ceil(x_max / bin_size))
    edges = np.linspace(0.0, n_bins * bin_size, n_bins + 1)

    fig, ax = plt.subplots()
    stats_per_run = {}
    all_means = []

    for label, df in items:
        if not (isinstance(df, pd.DataFrame) and x_col in df.columns and y_col in df.columns):
            continue

        sub = df[[x_col, y_col]].dropna()
        sub = sub[(sub[x_col] >= 0) & (sub[x_col] <= x_max)]
        if sub.empty:
            continue

        # ---- IMPORTANT: qui garantiamo che si chiami "bin" ----
        bin_s = pd.cut(sub[x_col].to_numpy(), bins=edges, include_lowest=True, right=True)
        bin_s = pd.Series(bin_s, name="bin")   # <<< FIX: nome colonna

        tmp = sub.copy()
        tmp["bin"] = bin_s

        g = tmp.groupby("bin", observed=True)[y_col]
        agg = g.agg(['mean', 'std', 'count']).reset_index()  # ora ha sempre colonna "bin"

        # centri bin
        centers = np.array([(iv.left + iv.right) / 2.0 for iv in agg["bin"]], dtype=float)
        agg["t"] = centers

        agg = agg.sort_values("t")
        agg = agg[agg["count"] > 0].reset_index(drop=True)

        if ema_alpha is not None and 0 < ema_alpha < 1 and not agg.empty:
            agg["mean"] = agg["mean"].ewm(alpha=ema_alpha, adjust=False).mean()

        if not agg.empty:
            ax.plot(agg["t"].to_numpy(), agg["mean"].to_numpy(), label=label)
            stats_per_run[label] = agg[["t", "mean", "std", "count"]].copy()
            all_means.append(agg["mean"].to_numpy())

    # ---- limiti Y ----
    if y_min is not None or y_max is not None:
        ax.set_ylim(bottom=y_min, top=y_max)
    else:
        if all_means:
            y_all = np.concatenate(all_means)
            y0, y1 = float(np.nanmin(y_all)), float(np.nanmax(y_all))
            if y0 == y1:
                delta = y0 * 0.01 if y0 != 0 else 1e-6
                y0, y1 = y0 - delta, y1 + delta
            pad = (y1 - y0) * float(y_pad_frac)
            ax.set_ylim(y0 - pad, y1 + pad)

    # ---- estetica ----
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_xlim(0, x_max)
    if grid:
        ax.grid(True, linestyle="--", alpha=0.5)
    if legend:
        ax.legend(title="Seed")

    if show:
        plt.show()

    return fig, ax, stats_per_run


#----------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Mapping, Iterable, Any

def _avgN_per_bin_from_events(in_times, out_times, *, bin_size, x_max=None):
    times = np.concatenate([in_times, out_times])
    deltas = np.concatenate([np.ones_like(in_times), -np.ones_like(out_times)])
    ok = ~np.isnan(times)
    times, deltas = times[ok], deltas[ok]
    if times.size == 0:
        return pd.DataFrame(columns=["t", "meanN", "count"])

    order = np.lexsort((deltas, times))  # risolve i tie: uscite prima
    times, deltas = times[order], deltas[order]
    max_time = float(times.max())
    if x_max is None or not np.isfinite(x_max):
        x_max = max_time
    x_max = max(x_max, 0.0)

    seg_t = np.concatenate([[0.0], times, [x_max]])
    seg_d = np.concatenate([[0], deltas, [0]])
    levels = np.cumsum(seg_d)[:-1]

    n_bins = int(np.ceil(x_max / bin_size))
    edges = np.linspace(0.0, n_bins * bin_size, n_bins + 1)
    num = np.zeros(n_bins)
    den = np.full(n_bins, bin_size)

    for i in range(len(levels)):
        t0, t1 = seg_t[i], seg_t[i+1]
        if t1 <= 0 or t0 >= x_max or t1 <= t0:
            continue
        t0c, t1c = max(t0, 0.0), min(t1, x_max)
        if t1c <= t0c:
            continue
        b0 = int(np.floor(t0c / bin_size))
        b1 = int(np.floor((t1c - 1e-12) / bin_size))
        for b in range(b0, b1 + 1):
            bin_start, bin_end = edges[b], edges[b+1]
            ovl = max(0.0, min(t1c, bin_end) - max(t0c, bin_start))
            if ovl > 0:
                num[b] += levels[i] * ovl

    centers = 0.5 * (edges[:-1] + edges[1:])
    meanN = num / den
    return pd.DataFrame({"t": centers, "meanN": meanN, "count": den})

def plot_avgN_system_per_run(
    dfs_runs,
    *,
    run_seeds=None,
    bin_size=100.0,
    x_max=None,
    normalize_time=False,   # <<< RIPRISTINATO
    ema_alpha=None,
    ema_min_periods=1,
    title="Numero medio di job nel sistema (transitorio)",
    xlabel=None,
    ylabel="N medio nel sistema",
    legend=True,
    grid=True,
    show=True,
):
    items = _normalize_items_with_seeds(dfs_runs, run_seeds)
    if not items:
        raise ValueError("dfs_runs vuoto")

    # x_max
    if x_max is None:
        tmax=[]
        for _,df in items:
            cols=[c for c in df.columns if c.startswith("IN_") or c.startswith("OUT_")]
            arr=pd.concat([df[c] for c in cols]).to_numpy().astype(float)
            if arr.size:
                tmax.append(np.nanmax(arr))
        if not tmax:
            raise ValueError("Nessuna colonna IN_/OUT_ trovata")
        x_max=float(max(tmax))

    fig,ax=plt.subplots()
    stats_per_run = {}

    for label,df in items:
        in_cols=[c for c in df.columns if c.startswith("IN_")]
        out_cols=[c for c in df.columns if c.startswith("OUT_")]

        in_times=pd.concat([df[c] for c in in_cols]).astype(float).to_numpy()
        out_times=pd.concat([df[c] for c in out_cols]).astype(float).to_numpy()

        agg=_avgN_per_bin_from_events(in_times,out_times,bin_size=bin_size,x_max=x_max)
        if agg.empty:
            continue

        x=agg["t"].to_numpy()
        y=agg["meanN"].to_numpy()

        # EMA
        if ema_alpha and 0<ema_alpha<1:
            y=pd.Series(y).ewm(alpha=ema_alpha,adjust=False,
                                min_periods=ema_min_periods).mean().to_numpy()

        # normalizzazione tempo
        if normalize_time and x_max>0:
            x = x / x_max
            xlabel_eff = "Tempo normalizzato"
        else:
            xlabel_eff = xlabel if xlabel else "Tempo (s)"

        ax.plot(x,y,label=label)

        stats_per_run[label] = pd.DataFrame({
            "t": agg["t"],
            "meanN": y
        })

    ax.set_title(title)
    ax.set_xlabel(xlabel_eff)
    ax.set_ylabel(ylabel)
    if grid:
        ax.grid(True, linestyle="--", alpha=0.5)
    if legend:
        ax.legend(title="Seed")

    if show:
        plt.show()

    return fig, ax, stats_per_run




In [16]:
fig, ax, thr_stats = plot_throughput_per_scenario(
    dfs_jobs,
    x_col="Completion",
    bin_size=800,          
    x_max=1000000,
    ema_alpha=0.01,        
    per_unit="per_second", 
    title="Throughput (completamenti/bin)"
)

fig, ax, util_stats = plot_utilization_per_scenario(
    dfs_jobs,
    servers=("A","B","P"),
    bin_size=800,
    x_max=1000000,
    ema_alpha=0.02,   
    title="Utilizzazione per scenario e server"
)

fig, ax, stats = plot_rt_mean_per_scenario(
    dfs_jobs, x_col="Completion", y_col="TempoRispostaTotal",
    bin_size=800, x_max=1000000, ema_alpha=0.02,y_pad_frac=1
  
)

In [11]:


fig, ax = plot_avgN_system_per_run(
    dfs_jobs,
    bin_size=800.0,
    normalize_time=False,
    ema_alpha=0.02,          # smoothing (più alto = più “reattivo”)
)


## Simulazione ad orizzonte finito con modello base e lbd variabile

## Generatore dei tassi lambda per la simulazione a orizzonte finito

In [1]:


# Fasce orarie giornaliere
fasce = ["00-06", "06-09", "09-12", "12-17", "17-21", "21-24"]

# Giorni della settimana
giorni = ["Sabato", "Domenica"]



def genera_lambda_giornaliero():
    """
    Genera un dizionario di lambda per fascia oraria in un giorno.
    I valori sono compresi tra 0.5 e 1.4 con picchi intorno a 1.4.
    """
    profilo = {}
    for fascia in fasce:
        if fascia in ["09-12", "17-21"]:
            # Fasce di picco (mattina e tardo pomeriggio)
            profilo[fascia] = round(Uniform(1.2, 1.4), 3)
        elif fascia in ["06-09", "12-17"]:
            # Fasce di medio traffico
            profilo[fascia] = round(Uniform(0.8, 1.1), 3)
        else:
            # Notte e tarda sera (basso carico)
            profilo[fascia] = round(Uniform(0.5, 0.7), 3)
    return profilo

def genera_lambda_weekend():
    """
    Genera un profilo per Sabato/Domenica:
    - notte e tarda sera: lambda basso (< 1.2)
    - fascia di punta 12-17: lambda tra 1.2 e 1.4
    - 17-21: attenuazione (medio)
    - mattina: medio-basso
    """
    profilo = {}
    for fascia in fasce:
        if fascia == "12-17":
            # Fascia di punta (12-18 circa): 1.2 - 1.4
            profilo[fascia] = round(Uniform(1.35, 1.4), 3)
        elif fascia == "17-21":
            # Dopo il picco: attenuazione
            profilo[fascia] = round(Uniform(1.2, 1.27), 3)
        elif fascia in ["06-09", "09-12"]:
            # Mattina/Mid: medio
            profilo[fascia] = round(Uniform(1.2, 1.3), 3)
        else:
            # Notte e tarda sera (00-06, 21-24): basso carico < 1.2
            profilo[fascia] = round(Uniform(1.1, 1.2), 3)
    return profilo


def genera_lambda_settimanale():
    """
    Genera un dizionario solo per Sabato e Domenica.
    """
    settimana = {}
    for giorno in giorni:  # ["Sabato", "Domenica"]
        settimana[giorno] = genera_lambda_weekend()
    return settimana


## Simulazione che lavora in ram

In [16]:

dfs_jobs = {}
dfs_seeds = {}

plantSeeds(12348948)


settimana=genera_lambda_settimanale()

print(settimana)

num_runs = 5
for i in range(num_runs):
    print(f"--- RUN {i} seeds PRIMA ---")
    _ = print_all_streams() 
    dfs_seeds[i] = snapshot_streams()    
    df_jobs, stats = simulate_ps_ABAPA_fast(
        lmb=0.5,
        mu_A1=5.0, mu_B=1.25, mu_A2=2.5, mu_P=1.42, mu_A3=6.67,
        threshold_time=172800)
    dfs_jobs[i] = df_jobs

    print(f"--- RUN {i} seeds DOPO ---")    
    _ = print_all_streams()               


write_streams_csv("PMCSN/seed.csv", dfs_seeds, run_field="RUN", encoding="utf-8")


In [12]:
dfs_seeds = {}

plantSeeds(12348948)

settimana = genera_lambda_settimanale()
print(settimana)

## Simulazione a orizzonte finito ch esalva ogni run nei file csv

In [13]:
import os
from pathlib import Path
import gc
import pandas as pd


num_runs = 5

out_dir = Path("PMCSN/Simulazioni")
out_dir.mkdir(parents=True, exist_ok=True)

for i in range(num_runs):
    print(f"--- RUN {i} seeds PRIMA ---")
    _ = print_all_streams()
    
    dfs_seeds[i] = snapshot_streams()
    
    df_jobs, stats = simulate_ps_ABAPA_fast(
        lmb=0.5,
        mu_A1=5.0, mu_B=1.25, mu_A2=2.5, mu_P=1.42, mu_A3=6.67,
        threshold_time=172800,
       
    )

    # ======= SCRIVI SUBITO SU DISCO E LIBERA MEMORIA =======
    # CSV
    df_jobs.to_csv(out_dir / f"jobs_run_{i:03d}.csv", index=False)
    # oppure PARQUET (più compatto):
    # df_jobs.to_parquet(out_dir / f"jobs_run_{i:03d}.parquet", index=False)

    del df_jobs
    gc.collect()
    # =====================================

    print(f"--- RUN {i} seeds DOPO ---")
    _ = print_all_streams()

# seeds tutti insieme
write_streams_csv(
    "PMCSN/seed.csv",
    dfs_seeds,
    run_field="RUN",
    encoding="utf-8"
)


In [55]:
run_seeds = [12348948, 1243098689, 628752245, 113235344, 637408609]

fig, ax, thr_stats = plot_throughput_per_scenario(
    dfs_jobs,
    run_seeds=run_seeds,
    x_col="Completion",
    bin_size=500,          
    x_max=172800,
    ema_alpha=0.01,        
    per_unit="per_second", 
    title="Throughput (completamenti/bin)"
)

fig, ax = plot_utilization_per_scenario(
    dfs_jobs,
    run_seeds=run_seeds,
    servers=("A","B","P"),
    bin_size=500,
    x_max=172800,
    ema_alpha=0.02,   
    title="Utilizzazione per scenario e server"
)

fig, ax ,_= plot_rt_mean_per_scenario(
    dfs_jobs, x_col="Completion", y_col="TempoRispostaTotal", run_seeds = run_seeds,
    bin_size=500, x_max=172800, ema_alpha=0.01,y_pad_frac=1
  
)

fig, ax, N_stats = plot_avgN_system_per_run(
    dfs_jobs,
    run_seeds=run_seeds,
    bin_size=800.0,
    normalize_time=False,
    ema_alpha=0.02,
    x_max=171000  
)



# Funzioni per stampare grafici per fasce versione csv // decide quale path usare per graficare base o migliorativo

In [32]:
# ================== DRAG & PLAY (A, B, P) con CSV + IC ==================
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- PARAMETRI ----------
BINS_PER_FASCIA = 24      # es. 24 bin su 6 ore = 15' a bin
SEC_PER_DAY     = 86400

# t*∞ per IC ~95% (approssimazione normale, coerente con formula n)
CONF_T_INF      = 1.96

# cartella dove hai salvato le simulazioni (un CSV per run)
SIM_DIR = "/home/luca/PMCSN/SimulazioniModelloMigliorativo"

# giorni / fasce (se non definiti altrove, uso default)
giorni = globals().get("giorni", ["Sabato","Domenica"])
fasce  = globals().get("fasce", ["00-06","06-09","09-12","12-17","17-21","21-24"])

# Mappa server -> visite da aggregare (A = A1,A2,A3)
SERVER_VISITS = {
    "A": [1, 3, 5],
    "B": [2],
    "P": [4],
}

# ---------- UTILS: LETTURA CSV ----------
def list_simulation_files(sim_dir=SIM_DIR):
    """
    Restituisce la lista ordinata dei file CSV di simulazione.
    Adatta il pattern se i file hanno un nome particolare (es. 'jobs_run_*.csv').
    """
    pattern = os.path.join(sim_dir, "*.csv")
    files = sorted(glob.glob(pattern))
    if not files:
        raise RuntimeError(f"Nessun CSV trovato in '{sim_dir}'. Controlla percorso e nomi file.")
    return files

# ---------- METRICHE DI BASE SU UNA SINGOLA RUN ----------
def mean_response_in_window(df, start, end):
    """
    Tempo di risposta medio (Completion - Arrival0) per job con Arrival0 in [start, end).
    Se esiste 'TempoRispostaTotal', uso quella colonna.
    """
    if not {"Arrival0", "Completion"}.issubset(df.columns):
        return np.nan
    
    mask = (df["Arrival0"] >= start) & (df["Arrival0"] < end)
    sub = df.loc[mask]
    if sub.empty:
        return np.nan
    
    if "TempoRispostaTotal" in sub.columns:
        return np.nanmean(sub["TempoRispostaTotal"].to_numpy(float))
    return np.nanmean((sub["Completion"] - sub["Arrival0"]).to_numpy(float))

def mean_jobs_in_system(df, start, end):
    """
    Numero medio di job nel sistema in [start, end).
    Approssimo integrando il numero di job in vita su quell'intervallo
    e dividendo per la lunghezza.
    """
    if not {"Arrival0", "Completion"}.issubset(df.columns):
        return np.nan
    
    interval = end - start
    if interval <= 0:
        return np.nan
    
    arr  = df["Arrival0"].to_numpy(float)
    comp = df["Completion"].to_numpy(float)
    overlaps = np.maximum(0.0, np.minimum(comp, end) - np.maximum(arr, start))
    return overlaps.sum() / interval

def utilization_server(df, visits, start, end):
    """
    Utilizzazione del SERVER nella finestra [start, end):
    frazione di tempo in cui il server è occupato (almeno 1 job),
    aggregando le visite indicate in `visits`.

    ATTENZIONE:
    - qui calcoliamo la **unione** degli intervalli di servizio,
      non la somma: in PS se ci sono più job contemporaneamente,
      il server è comunque "busy" una sola volta.
    """
    interval = end - start
    if interval <= 0:
        return np.nan

    intervals = []

    # Raccogli tutti gli intervalli [IN_k, OUT_k] che intersecano [start, end)
    for k in visits:
        in_col, out_col = f"IN_{k}", f"OUT_{k}"
        if in_col in df.columns and out_col in df.columns:
            IN  = df[in_col].to_numpy(float)
            OUT = df[out_col].to_numpy(float)

            s = np.maximum(IN, start)
            e = np.minimum(OUT, end)
            mask = e > s  # solo intervalli che hanno overlap
            for si, ei in zip(s[mask], e[mask]):
                intervals.append((si, ei))

    if not intervals:
        return np.nan

    # Ordina e unisci gli intervalli (union of intervals)
    intervals.sort(key=lambda x: x[0])
    merged = []
    cur_s, cur_e = intervals[0]

    for s, e in intervals[1:]:
        if s <= cur_e:
            # sovrapposto o adiacente: estendi
            cur_e = max(cur_e, e)
        else:
            # disgiunto: chiudi il precedente e apri uno nuovo
            merged.append((cur_s, cur_e))
            cur_s, cur_e = s, e
    merged.append((cur_s, cur_e))

    busy_total = sum(e - s for s, e in merged)

    # Utilizzazione = tempo totale in cui il server è busy / durata finestra
    rho = busy_total / interval
    return rho


# ---------- COSTRUZIONE FASCE ----------
def build_fasce_critiche_df_from_map(fasce_map, giorni, settimana=None, sec_per_day=SEC_PER_DAY):
    """
    Crea un DataFrame con:
      - giorno
      - fascia_id (1,2,...)
      - fascia (etichetta tipo "09-12")
      - lambda (se settimana è un dict annidato giorno->fascia->lambda)
      - start_s, end_s in secondi dall'inizio della settimana.
    """
    # normalizza: accetta dict {id:"HH-HH"} o lista/seq ["HH-HH", ...]
    if isinstance(fasce_map, dict):
        fasce_dict = {int(k): str(v) for k, v in fasce_map.items()}
    else:
        fasce_dict = {i + 1: str(v) for i, v in enumerate(list(fasce_map))}
    
    rows = []
    for day_idx, g in enumerate(giorni):
        base = day_idx * sec_per_day
        for fid, label in fasce_dict.items():
            h1, h2 = map(int, label.split("-"))
            start_s = base + h1 * 3600
            end_s   = base + h2 * 3600
            lam = np.nan
            if isinstance(settimana, dict):
                lam = float(settimana.get(g, {}).get(label, np.nan))
            rows.append({
                "giorno": g,
                "fascia_id": fid,
                "fascia": label,
                "lambda": lam,
                "start_s": start_s,
                "end_s": end_s
            })
    
    return (
        pd.DataFrame(rows)
        .sort_values(["fascia_id", "start_s"])
        .reset_index(drop=True)
    )

# ---------- SUPPORTO BINNING ----------
def _bin_edges(start_s, end_s, n_bins):
    return np.linspace(start_s, end_s, n_bins + 1)

def _bin_mid_rel_hours(edges, fascia_start_s):
    mids = 0.5 * (edges[:-1] + edges[1:])
    return (mids - fascia_start_s) / 3600.0  # ore relative 0..durata_fascia

def _avg_over_week_for_run(df_run, righe_fascia, fn_metric, ref_edges):
    """
    Per una singola run:
      - righe_fascia: tutte le 7 righe (una per giorno) di una certa fascia (es. 09-12)
      - ref_edges: edges della fascia del primo giorno
    Restituisce un array y[bin] = media sul week della metrica.
    """
    y = np.full(len(ref_edges) - 1, np.nan, dtype=float)
    s_ref = righe_fascia.iloc[0].start_s
    vals_per_bin = [[] for _ in range(len(ref_edges) - 1)]
    
    # scorro i 7 giorni e allineo i bin
    for r in righe_fascia.itertuples():
        shift = r.start_s - s_ref
        edges_day = ref_edges + shift
        for b in range(len(ref_edges) - 1):
            s_bin, e_bin = edges_day[b], edges_day[b + 1]
            vals_per_bin[b].append(fn_metric(df_run, s_bin, e_bin))
    
    for b in range(len(ref_edges) - 1):
        v = vals_per_bin[b]
        y[b] = np.nanmean(v) if v else np.nan
    return y

# ---------- INTERVALLI DI CONFIDENZA SULLE REPLICHE ----------
def _profiles_with_ci_from_csvs(righe_fascia, fn_metric, ref_edges, sim_dir=SIM_DIR):
    """
    Per una fascia e una metrica:
      - legge tutti i CSV in sim_dir
      - per ogni run calcola il profilo settimanale mediato sulla fascia
      - per ogni bin costruisce media e IC 95% usando le repliche come campione.

    IC costruito come:
      w_b = t*∞ * S_b / sqrt(n_b - 1)
      IC_b : mean_b ± w_b
    """
    files = list_simulation_files(sim_dir)
    
    n_bins = len(ref_edges) - 1
    vals_per_bin = [[] for _ in range(n_bins)]
    
    # accumulo i valori per bin su tutte le repliche
    for fp in files:
        df_run = pd.read_csv(fp)
        y = _avg_over_week_for_run(df_run, righe_fascia, fn_metric, ref_edges)
        for b in range(n_bins):
            if not np.isnan(y[b]):
                vals_per_bin[b].append(float(y[b]))
    
    mean_vals  = np.full(n_bins, np.nan)
    lower_vals = np.full(n_bins, np.nan)
    upper_vals = np.full(n_bins, np.nan)
    
    for b in range(n_bins):
        vals = np.array(vals_per_bin[b], dtype=float)
        if vals.size == 0:
            continue
        
        m = vals.mean()
        mean_vals[b] = m
        
        if vals.size > 1:
            s = vals.std(ddof=1)
            n = vals.size
            # semiampiezza: w = t*∞ * s / sqrt(n-1)
            h = CONF_T_INF * s / np.sqrt(n - 1)
            lower_vals[b] = m - h
            upper_vals[b] = m + h
        else:
            # con 1 sola replica: niente IC sensata
            lower_vals[b] = np.nan
            upper_vals[b] = np.nan
    
    return mean_vals, lower_vals, upper_vals

# ---------- PLOT CON IC ----------
def plot_profiles_with_ci_from_csvs(fasce_critiche_df, bins_per_fascia=BINS_PER_FASCIA, sim_dir=SIM_DIR):
    """
    Per ogni fascia:
      - costruisce i bin
      - per ciascuna metrica (tempo risposta, job in sistema, utilizzazione A/B/P)
        calcola la media sui run e l'IC 95% per ogni bin
      - plotta curva media + banda di IC.
    """
    fasce_uniche = fasce_critiche_df["fascia"].unique()
    
    for fascia in fasce_uniche:
        righe = fasce_critiche_df[fasce_critiche_df["fascia"] == fascia].sort_values("start_s")
        f_start, f_end = righe.iloc[0].start_s, righe.iloc[0].end_s
        edges   = _bin_edges(f_start, f_end, bins_per_fascia)
        x_rel_h = _bin_mid_rel_hours(edges, f_start)

        # 1) Tempo di risposta medio
        mean_y, low_y, up_y = _profiles_with_ci_from_csvs(
            righe_fascia=righe,
            fn_metric=mean_response_in_window,
            ref_edges=edges,
            sim_dir=sim_dir
        )
        plt.figure(figsize=(9, 5))
        plt.plot(x_rel_h, mean_y, label="Media repliche")
        if not np.all(np.isnan(low_y)):
            plt.fill_between(x_rel_h, low_y, up_y, alpha=0.25, label="IC 95%")
        plt.title(f"Fascia {fascia} – Tempo di risposta medio (media + IC 95%)")
        plt.xlabel("Ore dall'inizio fascia")
        plt.ylabel("Tempo di risposta medio (s)")
        plt.grid(alpha=0.3, linestyle="--")
        plt.legend()
        plt.tight_layout()
        plt.show()

        # 2) Job nel sistema
        mean_y, low_y, up_y = _profiles_with_ci_from_csvs(
            righe_fascia=righe,
            fn_metric=mean_jobs_in_system,
            ref_edges=edges,
            sim_dir=sim_dir
        )
        plt.figure(figsize=(9, 5))
        plt.plot(x_rel_h, mean_y, label="Media repliche")
        if not np.all(np.isnan(low_y)):
            plt.fill_between(x_rel_h, low_y, up_y, alpha=0.25, label="IC 95%")
        plt.title(f"Fascia {fascia} – Job nel sistema (media + IC 95%)")
        plt.xlabel("Ore dall'inizio fascia")
        plt.ylabel("Job simultanei medi")
        plt.grid(alpha=0.3, linestyle="--")
        plt.legend()
        plt.tight_layout()
        plt.show()

        # 3) Utilizzazione per SERVER (A, B, P)
        for srv, visits in SERVER_VISITS.items():
            mean_y, low_y, up_y = _profiles_with_ci_from_csvs(
                righe_fascia=righe,
                fn_metric=lambda dfr, s, e, _vis=visits: utilization_server(dfr, _vis, s, e),
                ref_edges=edges,
                sim_dir=sim_dir
            )
            plt.figure(figsize=(9, 5))
            plt.plot(x_rel_h, mean_y, label="Media repliche")
            if not np.all(np.isnan(low_y)):
                plt.fill_between(x_rel_h, low_y, up_y, alpha=0.25, label="IC 95%")
            plt.title(f"Fascia {fascia} – Utilizzazione server {srv} (media + IC 95%)")
            plt.xlabel("Ore dall'inizio fascia")
            plt.ylabel("Utilizzazione")
            plt.grid(alpha=0.3, linestyle="--")
            plt.legend()
            plt.tight_layout()
            plt.show()

# ---------- BUILD + PLOT ----------
# settimana può essere un dict come prima, se vuoi visualizzare anche lambda
settimana = globals().get("settimana")

fasce_critiche_df = build_fasce_critiche_df_from_map(
    fasce_map=fasce,
    giorni=giorni,
    settimana=settimana
)

plot_profiles_with_ci_from_csvs(fasce_critiche_df, bins_per_fascia=BINS_PER_FASCIA, sim_dir=SIM_DIR)


# ================== END DRAG & PLAY ==================




# Per printare i grafici relativi alle perdite

In [16]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_loss_summary_from_stats(sim_dir=SIM_DIR):
    """
    Legge stats_all_runs.csv in sim_dir e plotta:
      - % job persi totale (media + IC 95%)
      - % job persi per server (media + IC 95%)
    """
    stats_path = os.path.join(sim_dir, "stats_all_runs.csv")
    if not os.path.exists(stats_path):
        print(f"[plot_loss_summary_from_stats] File stats_all_runs.csv non trovato in '{sim_dir}'.")
        return
    
    stats_all = pd.read_csv(stats_path)

    # ------------------ 1) % PERSI TOTALE ------------------
    if "pct_lost_total" in stats_all.columns:
        vals = stats_all["pct_lost_total"].to_numpy(float)
        m = np.mean(vals)
        if vals.size > 1:
            s = np.std(vals, ddof=1)
            n = vals.size
            # stessa logica degli altri IC: w = t_inf * s / sqrt(n-1)
            h = CONF_T_INF * s / np.sqrt(n - 1)
        else:
            h = np.nan

        plt.figure(figsize=(6, 5))
        plt.bar(["Totale"], [m])
        if not np.isnan(h):
            plt.errorbar(
                ["Totale"], [m],
                yerr=[h],
                fmt="none",
                capsize=5,
                label="IC 95%"
            )
        plt.ylabel("% job persi")
        plt.title("Percentuale di job persi (totale sulle repliche)")
        plt.grid(alpha=0.3, axis="y", linestyle="--")
        if not np.isnan(h):
            plt.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("[plot_loss_summary_from_stats] Colonna 'pct_lost_total' non trovata negli stats.")

    # ------------------ 2) % PERSI PER SERVER ------------------
    # cerco tutte le colonne che iniziano con "loss_pct_"
    loss_cols = [c for c in stats_all.columns if c.startswith("loss_pct_")]
    if not loss_cols:
        print("[plot_loss_summary_from_stats] Nessuna colonna 'loss_pct_*' trovata negli stats.")
        return

    servers = [c.replace("loss_pct_", "") for c in loss_cols]
    means = []
    errs  = []

    for c in loss_cols:
        vals = stats_all[c].to_numpy(float)
        m = np.mean(vals)
        if vals.size > 1:
            s = np.std(vals, ddof=1)
            n = vals.size
            h = CONF_T_INF * s / np.sqrt(n - 1)
        else:
            h = np.nan
        means.append(m)
        errs.append(h)

    x = np.arange(len(servers))

    plt.figure(figsize=(8, 5))
    plt.bar(x, means)
    # errorbar solo dove ha senso
    yerr = [e if not np.isnan(e) else 0.0 for e in errs]
    if not all(np.isnan(errs)):
        plt.errorbar(x, means, yerr=yerr, fmt="none", capsize=5, label="IC 95%")
    plt.xticks(x, servers)
    plt.ylabel("% job persi")
    plt.title("Percentuale di job persi per server (media + IC 95%)")
    plt.grid(alpha=0.3, axis="y", linestyle="--")
    if not all(np.isnan(errs)):
        plt.legend()
    plt.tight_layout()
    plt.show()

plot_loss_summary_from_stats()

# Funzioni per stampare grafici per fasce versione normale predo df dalla ram

In [12]:
# ================== DRAG & PLAY (A, B, P) ==================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- PARAMETRI ----------
BINS_PER_FASCIA = 24     # es. 24 bin su 6 ore = 15' a bin
SEC_PER_DAY     = 86400
giorni = globals().get("giorni", ["Sabato","Domenica"])
fasce  = globals().get("fasce", ["00-06","06-09","09-12","12-17","17-21","21-24"])

# Mappa server -> visite da aggregare (A = A1,A2,A3)
SERVER_VISITS = {
    "A": [1, 3, 5],
    "B": [2],
    "P": [4],
}

# ---------- CHECK DATI ----------
if "dfs_jobs" not in globals() or not isinstance(dfs_jobs, dict) or len(dfs_jobs) == 0:
    raise RuntimeError("Attenzione: serve dfs_jobs = {run_label: DataFrame}.")
num_runs = len(dfs_jobs)

# ---------- METRICHE ----------
def mean_response_in_window(df, start, end):
    if not {"Arrival0","Completion"}.issubset(df.columns):
        return np.nan
    mask = (df["Arrival0"] >= start) & (df["Arrival0"] < end)
    sub = df.loc[mask]
    if sub.empty: return np.nan
    if "TempoRispostaTotal" in sub:
        return np.nanmean(sub["TempoRispostaTotal"].to_numpy(float))
    return np.nanmean((sub["Completion"] - sub["Arrival0"]).to_numpy(float))

def mean_jobs_in_system(df, start, end):
    if not {"Arrival0","Completion"}.issubset(df.columns):
        return np.nan
    interval = end - start
    if interval <= 0: return np.nan
    arr  = df["Arrival0"].to_numpy(float)
    comp = df["Completion"].to_numpy(float)
    overlaps = np.maximum(0.0, np.minimum(comp, end) - np.maximum(arr, start))
    return overlaps.sum() / interval

def utilization_server(df, visits, start, end):
    """
    Utilizzazione del SERVER nella finestra, aggregando più visite.
    Calcolo: (somma dei tempi occupati di tutte le visite) / (durata finestra).
    """
    interval = end - start
    if interval <= 0:
        return np.nan
    busy_total = 0.0
    has_any = False
    for k in visits:
        in_col, out_col = f"IN_{k}", f"OUT_{k}"
        if in_col in df.columns and out_col in df.columns:
            IN  = df[in_col].to_numpy(float)
            OUT = df[out_col].to_numpy(float)
            busy = np.maximum(0.0, np.minimum(OUT, end) - np.maximum(IN, start)).sum()
            busy_total += busy
            has_any = True
    if not has_any:
        return np.nan
    return busy_total / interval

# ---------- COSTRUZIONE FASCE ----------
def build_fasce_critiche_df_from_map(fasce_map, giorni, settimana=None, sec_per_day=SEC_PER_DAY):
    # normalizza: accetta dict {id:"HH-HH"} o lista/seq ["HH-HH", ...]
    if isinstance(fasce_map, dict):
        fasce_dict = {int(k): str(v) for k, v in fasce_map.items()}
    else:
        fasce_dict = {i+1: str(v) for i, v in enumerate(list(fasce_map))}
    rows = []
    for day_idx, g in enumerate(giorni):
        base = day_idx * sec_per_day
        for fid, label in fasce_dict.items():
            h1, h2 = map(int, label.split("-"))
            start_s = base + h1 * 3600
            end_s   = base + h2 * 3600
            lam = np.nan
            if isinstance(settimana, dict):
                lam = float(settimana.get(g, {}).get(label, np.nan))
            rows.append({
                "giorno": g, "fascia_id": fid, "fascia": label,
                "lambda": lam, "start_s": start_s, "end_s": end_s
            })
    return pd.DataFrame(rows).sort_values(["fascia_id","start_s"]).reset_index(drop=True)

# ---------- SPAGHETTI PLOT ----------
def _bin_edges(start_s, end_s, n_bins): return np.linspace(start_s, end_s, n_bins + 1)
def _bin_mid_rel_hours(edges, fascia_start_s):
    mids = 0.5 * (edges[:-1] + edges[1:])
    return (mids - fascia_start_s) / 3600.0  # ore relative 0..durata_fascia

def _avg_over_week_for_run(df_run, righe_fascia, fn_metric, ref_edges):
    y = np.full(len(ref_edges) - 1, np.nan, dtype=float)
    s_ref = righe_fascia.iloc[0].start_s
    vals_per_bin = [[] for _ in range(len(ref_edges) - 1)]
    for r in righe_fascia.itertuples():
        shift = r.start_s - s_ref
        edges_day = ref_edges + shift
        for b in range(len(ref_edges) - 1):
            s_bin, e_bin = edges_day[b], edges_day[b + 1]
            vals_per_bin[b].append(fn_metric(df_run, s_bin, e_bin))
    for b in range(len(ref_edges) - 1):
        v = vals_per_bin[b]
        y[b] = np.nanmean(v) if v else np.nan
    return y

def plot_spaghetti_by_time(dfs_jobs, fasce_critiche_df, bins_per_fascia=BINS_PER_FASCIA):
    fasce_uniche = fasce_critiche_df["fascia"].unique()
    for fascia in fasce_uniche:
        righe = fasce_critiche_df[fasce_critiche_df["fascia"] == fascia].sort_values("start_s")
        f_start, f_end = righe.iloc[0].start_s, righe.iloc[0].end_s
        edges   = _bin_edges(f_start, f_end, bins_per_fascia)
        x_rel_h = _bin_mid_rel_hours(edges, f_start)

        # 1) Tempo di risposta medio
        plt.figure(figsize=(9, 5))
        for run_idx, (_, df_run) in enumerate(dfs_jobs.items()):
            y = _avg_over_week_for_run(df_run, righe, mean_response_in_window, edges)
            plt.plot(x_rel_h, y, label=f"Run {run_idx}")
        plt.title(f"Fascia {fascia} – Tempo di risposta medio (profilo nella fascia, media sui 7 giorni)")
        plt.xlabel("Ore dall'inizio fascia"); plt.ylabel("Tempo di risposta medio (s)")
        plt.grid(alpha=0.3, linestyle="--"); plt.legend(); plt.tight_layout(); plt.show()

        # 2) Job nel sistema
        plt.figure(figsize=(9, 5))
        for run_idx, (_, df_run) in enumerate(dfs_jobs.items()):
            y = _avg_over_week_for_run(df_run, righe, mean_jobs_in_system, edges)
            plt.plot(x_rel_h, y, label=f"Run {run_idx}")
        plt.title(f"Fascia {fascia} – Job nel sistema (profilo nella fascia, media sui 7 giorni)")
        plt.xlabel("Ore dall'inizio fascia"); plt.ylabel("Job simultanei medi")
        plt.grid(alpha=0.3, linestyle="--"); plt.legend(); plt.tight_layout(); plt.show()

        # 3) Utilizzazione per SERVER (A, B, P) — 1 grafico per server, 1 linea per run
        for srv, visits in SERVER_VISITS.items():
            plt.figure(figsize=(9, 5))
            for run_idx, (_, df_run) in enumerate(dfs_jobs.items()):
                y = _avg_over_week_for_run(
                    df_run, righe,
                    lambda dfr, s, e, _vis=visits: utilization_server(dfr, _vis, s, e),
                    edges
                )
                plt.plot(x_rel_h, y, label=f"Run {run_idx}")
            plt.title(f"Fascia {fascia} – Utilizzazione server {srv} (media settimanale, profilo nella fascia)")
            plt.xlabel("Ore dall'inizio fascia"); plt.ylabel("Utilizzazione")
            plt.grid(alpha=0.3, linestyle="--"); plt.legend(); plt.tight_layout(); plt.show()

# ---------- BUILD + PLOT ----------
fasce_critiche_df = build_fasce_critiche_df_from_map(
    fasce_map=fasce, giorni=giorni, settimana=globals().get("settimana")
)
plot_spaghetti_by_time(dfs_jobs, fasce_critiche_df, bins_per_fascia=BINS_PER_FASCIA)
# ================== END DRAG & PLAY ==================


# funzione riproducibilita dell repliche

In [14]:
import pandas as pd
import os

STREAM_NAME_TO_INDEX = {
    "ARR": 0,
    "A1":  1,
    "A2":  2,
    "A3":  3,
    "B":   4,
    "P":   5,
}

def restore_streams_from_row(row):
    """
    Prende una riga del CSV dei seed (per una certa RUN) e
    reimposta tutti gli stream RNG ai seed corrispondenti.
    """
    for name, idx in STREAM_NAME_TO_INDEX.items():
        if name not in row:
            raise ValueError(f"Colonna '{name}' non trovata nel CSV dei seed.")
        seed_val = int(row[name])
        selectStream(idx)
        putSeed(seed_val)

def replica_run_e_confronta(
    target_run: int,
    path_jobs_originale: str,
    path_seed_csv: str,
    settimana: dict,
    out_dir: str = "PMCSN",
    run_field: str = "RUN",
    lmb: float = 0.5,
    mu_A1: float = 5.0, 
    mu_B: float = 1.25,
    mu_A2: float = 2.5,
    mu_P: float = 2.5,
    mu_A3: float = 10.0,
    threshold_time: float = 172800.0,
    tol: float = 1e-10,     # tolleranza per i float
):
    """
    Replica una run della simulazione partendo dagli stessi seed e
    confronta il CSV originale con quello replicato (entro una certa tolleranza).
    """
    # 1) Carica il CSV originale della run da replicare
    df_orig = pd.read_csv(path_jobs_originale)

    # 2) Carica il CSV dei seed
    df_seeds = pd.read_csv(path_seed_csv)

    # 3) Usa i seed della STESSA run che vuoi replicare (NON la precedente)
    mask = df_seeds[run_field] == target_run
    if not mask.any():
        raise ValueError(
            f"Nessuna riga trovata in '{path_seed_csv}' per {run_field} = {target_run}"
        )
    row_run = df_seeds[mask].iloc[0]

    # 4) Ripristina gli stream RNG allo stato iniziale della run target_run
    restore_streams_from_row(row_run)

    print(f"--- REPLICA RUN {target_run}: seeds PRIMA ---")
    _ = print_all_streams()

    # 5) Esegui di nuovo la simulazione
    df_rep, stats_rep = simulate_ps_ABAPA_mod(
        lmb=lmb,
        mu_A1=mu_A1, mu_B=mu_B, mu_A2=mu_A2, mu_P=mu_P, mu_A3=mu_A3,
        threshold_time=threshold_time,
        settimana=settimana
    )

    print(f"--- REPLICA RUN {target_run}: seeds DOPO ---")
    _ = print_all_streams()

    # 6) Salva il CSV replicato
    os.makedirs(out_dir, exist_ok=True)
    path_replica = os.path.join(out_dir, f"jobs_run_{target_run}_replicata.csv")
    df_rep.to_csv(path_replica, index=False)
    print(f"CSV replicato salvato in: {path_replica}")

    # 7) Confronto struttura
    same_shape = (df_orig.shape == df_rep.shape)
    same_cols = (list(df_orig.columns) == list(df_rep.columns))

    if not same_shape or not same_cols:
        print("ATTENZIONE: forma o colonne diverse tra originale e replica.")
        print("shape originale:", df_orig.shape, "  shape replica:", df_rep.shape)
        print("colonne originale:", list(df_orig.columns))
        print("colonne replica:  ", list(df_rep.columns))
        sono_uguali = False
    else:
        # 8) Confronto con tolleranza
        #   - per le colonne numeriche: confronto a tolleranza
        #   - per le altre: equals secco
        num_cols = df_orig.select_dtypes(include=["number"]).columns.tolist()
        other_cols = [c for c in df_orig.columns if c not in num_cols]

        numeric_close = True
        max_diff = 0.0

        if num_cols:
            diff_abs = (df_orig[num_cols] - df_rep[num_cols]).abs()
            max_diff = float(diff_abs.to_numpy().max())
            numeric_close = (max_diff <= tol)
            print(f"Massima differenza numerica (|orig-rep|): {max_diff:.3e}")

        other_equal = True
        if other_cols:
            other_equal = df_orig[other_cols].equals(df_rep[other_cols])

        sono_uguali = numeric_close and other_equal

        if sono_uguali:
            print("✅ I due DataFrame sono uguali entro la tolleranza numerica.")
        else:
            print("❌ I due DataFrame differiscono oltre la tolleranza.")
            if not numeric_close:
                print(f"- Differenze numeriche superiori a {tol}")
            if not other_equal:
                print("- Differenze in colonne non numeriche.")

    return df_orig, df_rep, sono_uguali


In [15]:


df_o, df_r, ok = replica_run_e_confronta(
    target_run=250,
    path_jobs_originale="/home/luca/PMCSN/Simulazioni/jobs_run_250.csv",
    path_seed_csv="PMCSN/seedOrizzonteFinito.csv",
    settimana=settimana,
    out_dir="PMCSN"
)

print("Risultato verifica riproducibilità:", ok)


# Modello Migliorativo 

In [56]:
import os, csv
from collections.abc import Mapping

# === LISTA UNICA DI TUTTI GLI STREAM ===
# Adatta questa lista ai tuoi STREAM_XXX effettivi (17 o quanti sono)
ALL_STREAM_NAMES = [
    "STREAM_ARR",
    "STREAM_A1_A", "STREAM_A1_A2",
    "STREAM_A2_A", "STREAM_A2_A2",
    "STREAM_A3_A", "STREAM_A3_A2",
    "STREAM_B_1", "STREAM_B_2",
    "STREAM_P",
    "STREAM_SPILLOVER_A",      # se NON esiste, commenta questa riga
    "STREAM_PROBE",
    "STREAM_LB_A", "STREAM_LB_B",
    "STREAM_LOSS_B", "STREAM_LOSS_A2",
    "STREAM_LOSS_P", "STREAM_LOSS_EXIT",
]


def snapshot_streamsMigliorativo():
    """
    Restituisce un NUOVO dict con il seed corrente per OGNI stream in ALL_STREAM_NAMES.

    Struttura:
        {
            "STREAM_ARR":        seed,
            "STREAM_A1_A":       seed,
            "STREAM_A1_A2":      seed,
            ...
            "STREAM_LOSS_EXIT":  seed,
        }
    """
    out: dict[str, int] = {}
    for name in ALL_STREAM_NAMES:
        idx = globals()[name]       # prende il valore numerico di STREAM_XXX
        selectStream(idx)
        out[name] = getSeed()
    return out


def print_all_streamsMigliorativo():
    """
    Stampa (debug) e ritorna una copia nuova dei seed per ogni stream.
    """
    d = snapshot_streamsMigliorativo()
    for name in ALL_STREAM_NAMES:
        print(f"{name}: {d[name]}")
    return d


def write_streams_csvMigliorativo(
    path: str,
    data: Mapping,                # {run_id: {"STREAM_ARR":seed, "STREAM_A1_A":seed, ...}, ...}
    *,
    run_field: str = "RUN",
    columns: list[str] | None = None,
    encoding: str = "utf-8"
):
    """
    Scrive il CSV con struttura:

        RUN,STREAM_ARR,STREAM_A1_A,STREAM_A1_A2,...
        0,  ...,        ...,        ...,         ...
        1,  ...,        ...,        ...,         ...
        ...

    dove ogni colonna corrisponde a UNO stream fisico.
    """
    # normalizza in lista ordinata per run_id
    rows = [(run_id, inner) for run_id, inner in sorted(data.items())]

    if columns is None:
        columns = list(ALL_STREAM_NAMES)

    header = [run_field] + columns
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    with open(path, "w", encoding=encoding, newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        for run_id, inner in rows:
            row = [run_id]
            for col in columns:
                row.append(inner.get(col, ""))
            w.writerow(row)


In [57]:
# parametri globali per la perdita
LOSS_B = 6.0  # coefficiente p (modificare finché non esce una probabilità decente)
LOSS_P = 10.0
LOSS_P_EXIT = 4.0  # esponente della curva tra 2s e 4s
FLAG=True #Se flag =false le probabiloita di perdita non vengono simulate


def loss_probability_from_sojourn_B(sojourn_sec: float) -> float:
    """
    Probabilità di perdita in funzione del sojourn time (secondi).
    Regole:
      - 0 se t <= 3
      - ((t-2)/3)**LOSS_P per 3 < t < 6 (clamp [0,1])
      - 1 se t >= 6
    """
    try:
        t = float(sojourn_sec)
    except (TypeError, ValueError):
        return 0.0

    if t <= 4.0:
        return 0.0
    if t >= 6.0:
        return 1.0

    val = ((t - 4.0) / 2.0) ** float(LOSS_B)

    if val < 0.0:
        val = 0.0
    elif val > 1.0:
        val = 1.0
    return val


def loss_prob_class2_after_P(service_time_p: float) -> float:
    """
    p_loss per CLASSE 2 (dopo P), binaria:
      - 0 se tempo di servizio in P <= T_PAY
      - 1 se tempo di servizio in P >  T_PAY
    """
    try:
        t = float(service_time_p)
    except (TypeError, ValueError):
        return 0.0

    if t <= 10:
        return 0
    if t >= 12.0:
        return 1.0
    val = ((t - 10.0) / 2.0) ** float(LOSS_B)

    if val < 0.0:
        val = 0.0
    elif val > 1.0:
        val = 1.0
    return val

def loss_prob_class3_at_exit(sojourn_sec: float) -> float:
    return 0.0

In [58]:
# ==============================
#  STREAM IDs (esempio coerente)
# ==============================

# Arrivi
STREAM_ARR       = 0

# Servizio A1 (prima visita in A)
STREAM_A1_A      = 1   # A fisico
STREAM_A1_A2     = 2   # A2 scalato

# Servizio B
STREAM_B_1       = 3   # B fisico
STREAM_B_2       = 4   # B2 scalato

# Servizio A2 (seconda visita in A)
STREAM_A2_A      = 5
STREAM_A2_A2     = 6

# Servizio P
STREAM_P         = 7

# Servizio A3 (terza visita in A)
STREAM_A3_A      = 8
STREAM_A3_A2     = 9

# Probe / spillover / LB
STREAM_PROBE        = 10   # probe + decisioni P vs spillover
STREAM_LB_A         = 11   # load balancing A/A2
STREAM_LB_B         = 12   # load balancing B/B2
STREAM_SPILLOVER_A  = 13   # servizio dei job rientrati in A/A2

# Perdita: stream separati per ciascun confine
STREAM_LOSS_B    = 14
STREAM_LOSS_A2   = 15
STREAM_LOSS_P    = 16
STREAM_LOSS_EXIT = 17


# SImulazione con LMB Costante

In [59]:
def simulate_ps_ABAPA_scaled_AB_with_spillover_and_loss_tags(
    lmb,
    mu_A1, mu_B, mu_A2, mu_P, mu_A3,
    threshold_time,
    *,
    # thresholds per scale-out/in
    k_threshold_A_up=0.5, k_threshold_A_down=0.2,
    k_threshold_B_up=0.5, k_threshold_B_down=0.2,
    # probing/adattamento su P
    p_send_to_P_init=0.6, p_probe=0.45, window_size=6, L_target=8.0, gamma=0.25,
    # domanda media usata in spillover (mean dell'esponenziale dei job rientrati)
    spillover_demand_const=0.6,
    eps=1e-12
):
    # ===== helpers RNG su stream =====
    def U01_from(stream_id):
        selectStream(stream_id)
        u = random()
        if u <= 0.0: return 1e-15
        if u >= 1.0: return 1.0 - 1e-15
        return u

    # ===== server PS tag-based =====
    class PSServer:
        __slots__ = ("name","c","S","n","heap","N_time_accum","busy_time")
        def __init__(self,name,c=1.0):
            self.name=name; self.c=c
            self.S=0.0; self.n=0
            self.heap=[]                 # (F, jid, seg)
            self.N_time_accum=0.0        # ∫ N(t) dt
            self.busy_time=0.0           # ∫ 1{n>0} dt

    servers = {k:PSServer(k,1.0) for k in ("A","B","P")}

    # === scaling & routing dinamico (A/A2, B/B2) ===
    def choose_A_server(now):
        util_A  = servers["A"].busy_time / now if now > 0 else 0.0
        util_A2 = servers["A2"].busy_time / now if ("A2" in servers and now > 0) else 0.0
        if util_A > k_threshold_A_up:
            if "A2" not in servers:
                servers["A2"] = PSServer("A2", 1.0)
            invA  = 1.0/util_A  if util_A  > 0 else 1.0
            invA2 = 1.0/util_A2 if util_A2 > 0 else 1.0
            p_A = invA / (invA + invA2)
            return "A" if U01_from(STREAM_LB_A) < p_A else "A2"
        return "A"

    def choose_B_server(now):
        util_B  = servers["B"].busy_time / now if now > 0 else 0.0
        util_B2 = servers["B2"].busy_time / now if ("B2" in servers and now > 0) else 0.0
        if util_B > k_threshold_B_up:
            if "B2" not in servers:
                servers["B2"] = PSServer("B2", 1.0)
            invB  = 1.0/util_B  if util_B  > 0 else 1.0
            invB2 = 1.0/util_B2 if util_B2 > 0 else 1.0
            p_B = invB / (invB + invB2)
            return "B" if U01_from(STREAM_LB_B) < p_B else "B2"
        return "B"

    def stage_server_name(pos, now):
        # come nella fast: A, B, A, P, A
        if pos in (1,3,5): return choose_A_server(now)
        if pos == 2:       return choose_B_server(now)
        return "P"

    def check_scaledown(now):
        if now <= 0: return
        util_A = servers["A"].busy_time / now
        if "A2" in servers and servers["A2"].n == 0 and util_A < k_threshold_A_down:
            del servers["A2"]
        util_B = servers["B"].busy_time / now
        if "B2" in servers and servers["B2"].n == 0 and util_B < k_threshold_B_down:
            del servers["B2"]

    # === sequenza visite & draw demand ===
    def a_visit_index_after(pos): 
        return [1,1,2,2,3][pos-1]

    # <<< CAMBIATA: dipende anche da server_name, usa i nuovi stream
    def draw_demand_for(pos, server_name):
        # pos: 1..5
        if pos == 1:
            # prima visita in A (A1)
            if server_name == "A":
                selectStream(STREAM_A1_A)
            else:  # "A2"
                selectStream(STREAM_A1_A2)
            return Exponential(1/mu_A1)

        elif pos == 2:
            # visita in B
            if server_name == "B":
                selectStream(STREAM_B_1)
            else:  # "B2"
                selectStream(STREAM_B_2)
            return Exponential(1/mu_B)

        elif pos == 3:
            # seconda visita in A (A2)
            if server_name == "A":
                selectStream(STREAM_A2_A)
            else:  # "A2"
                selectStream(STREAM_A2_A2)
            return Exponential(1/mu_A2)

        elif pos == 4:
            # P (unico)
            selectStream(STREAM_P)
            return Exponential(1/mu_P)

        else:  # pos == 5
            # terza visita in A (A3)
            if server_name == "A":
                selectStream(STREAM_A3_A)
            else:  # "A2"
                selectStream(STREAM_A3_A2)
            return Exponential(1/mu_A3)
    # >>> FINE draw_demand_for

    # ===== stato globale =====
    t=0.0
    selectStream(STREAM_ARR)
    mean_inter=1.0/float(lmb)
    next_arrival = t + Exponential(mean_inter)

    job_id=0
    jobs_rows=[]
    jobs_info={}
    all_stage_records = []

    p_send_to_P = float(p_send_to_P_init)
    obs_lat_P=[]

    arrivals_by_server, losses_by_server = {}, {}
    lost_total = loss_on_b = loss_on_p = loss_on_a2 = 0
    total_arrivals = 0

    def bump_arrival(s): arrivals_by_server[s]=arrivals_by_server.get(s,0)+1
    def bump_loss(s):    losses_by_server[s]=losses_by_server.get(s,0)+1

    # === advance: IDENTICO alla fast ma con busy_time ===
    def advance(delta):
        nonlocal t
        if delta <= 0: return
        for srv in servers.values():
            srv.N_time_accum += srv.n * delta
            if srv.n>0: srv.busy_time += delta
        for srv in servers.values():
            if srv.n>0:
                srv.S += (srv.c / srv.n) * delta
        t += delta

    # ammissione segmento
    # <<< CAMBIATA: passa server_name a draw_demand_for
    def admit_segment(jid,pos,server_name,arrival_t,demand=None):
        srv = servers[server_name]
        d = draw_demand_for(pos, server_name) if demand is None else demand
        n_at_admit = srv.n + 1
        seg = {
            "job_id": jid, "pos": pos, "server": server_name,
            "a_visit": a_visit_index_after(pos),
            "arrival_stage": arrival_t, "IN": arrival_t,
            "S_admit": srv.S, "Demand": d, "N_at_Admit": n_at_admit
        }
        F = seg["S_admit"] + d
        heapq.heappush(srv.heap, (F, jid, seg))
        srv.n += 1
        return seg
    # >>> FINE admit_segment

    INF = math.inf
    def arrivals_open(): 
        return (next_arrival is not None and next_arrival <= threshold_time)
    def work_left():
        return arrivals_open() or any(srv.n>0 for srv in servers.values())

    while work_left():
        # prossimo completion (regola tag-based)
        next_comp_time = INF; comp_srv = None
        for srv in servers.values():
            if srv.n==0 or not srv.heap: 
                continue
            F_min, _, _ = srv.heap[0]
            if F_min <= srv.S + 1e-15:
                dt_srv = 0.0
            else:
                dS = F_min - srv.S
                dt_srv = dS * (srv.n / srv.c)
            tc = t + dt_srv
            if tc < next_comp_time:
                next_comp_time = tc; comp_srv = srv

        ta = next_arrival if arrivals_open() else INF
        t_next = min(ta, next_comp_time)
        if math.isinf(t_next): break

        advance(t_next - t)
        check_scaledown(t)

        if abs(t_next - ta) <= 1e-12:
            # ARRIVO
            job_id += 1; jid = job_id; total_arrivals += 1
            jobs_info[jid] = {
                "arrival0": t, "stage_recs": [], "demand_total": 0.0,
                "spillover_used": False, "is_probe": (U01_from(STREAM_PROBE) < p_probe)
            }
            pos = 1
            server_name = stage_server_name(pos, t)
            admit_segment(jid, pos, server_name, t)
            bump_arrival(server_name)

            selectStream(STREAM_ARR)
            ia = Exponential(mean_inter)
            cand = t + ia
            next_arrival = cand if cand <= threshold_time else None

        else:
            # COMPLETION
            F_min, jid, seg = heapq.heappop(comp_srv.heap)
            comp_srv.n -= 1

            IN = seg["IN"]; OUT = t
            Demand = seg["Demand"]; Wait = 0.0
            Service = OUT - IN
            n_at_admit = seg["N_at_Admit"]; avgN_during = n_at_admit

            rec = {
                "pos": seg["pos"], "server": seg["server"], "id_label": f"{jid}[{seg['a_visit']}]",
                "IN": IN, "OUT": OUT, "Demand": Demand, "Wait": Wait,
                "Service": Service, "N_at_Admit": n_at_admit, "AvgN_during": avgN_during
            }
            job = jobs_info[jid]
            job["stage_recs"].append(rec)
            all_stage_records.append(rec)  # <== per metriche per visita
            job["demand_total"] += Demand

            # feedback su P (solo probe)
            if seg["server"] == "P" and job.get("is_probe", False):
                obs_lat_P.append(Service + Wait)
                if len(obs_lat_P) > window_size: obs_lat_P.pop(0)
                if len(obs_lat_P) >= window_size:
                    Lhat = sum(obs_lat_P)/len(obs_lat_P)
                    if Lhat > L_target: p_send_to_P = max(0.0, p_send_to_P*(1-gamma))
                    else:               p_send_to_P = min(1.0, p_send_to_P*(1+gamma))

            # === perdite ai confini, rispettando FLAG (stream dedicati) ===
            lost_here = False
            if FLAG:
                if seg["pos"] == 2:
                    p_loss = float(loss_probability_from_sojourn_B(Service))
                    lost_here = (U01_from(STREAM_LOSS_B) < p_loss)
                    if lost_here: loss_on_b += 1
                elif seg["pos"] == 3 and not job["spillover_used"]:
                    p_loss = 0.1
                    lost_here = (U01_from(STREAM_LOSS_A2) < p_loss)
                    if lost_here: loss_on_a2 += 1
                elif seg["pos"] == 4:
                    p_loss = float(loss_prob_class2_after_P(Service))
                    lost_here = (U01_from(STREAM_LOSS_P) < p_loss)
                    if lost_here: loss_on_p += 1
                elif seg["pos"] == 5:
                    p_loss = float(loss_prob_class3_at_exit(Service))
                    lost_here = (U01_from(STREAM_LOSS_EXIT) < p_loss)

            if lost_here:
                lost_total += 1
                bump_loss(seg["server"])
                continue

            # routing o fine
            if seg["pos"] < 5:
                pos2 = seg["pos"] + 1
                if pos2 == 4:
                    # decide spillover vs P con lo stream delle probe
                    do_spill = (U01_from(STREAM_PROBE) > p_send_to_P)
                    if do_spill and (not job["spillover_used"]):
                        # SPILLOVER: rientro in A/A2 con stream di servizio dedicato
                        server2 = stage_server_name(3, t)   # A/A2 fisici
                        selectStream(STREAM_SPILLOVER_A)
                        demand2 = Exponential(spillover_demand_const)  # mean = spillover_demand_const
                        pos2 = seg["pos"]                  # resta nello stadio "3" (A2 logico)
                        job["spillover_used"] = True
                    else:
                        server2 = "P"
                        demand2 = draw_demand_for(4, server2)
                        pos2 = 4
                else:
                    server2 = stage_server_name(pos2, t)
                    demand2 = draw_demand_for(pos2, server2)

                admit_segment(jid, pos2, server2, t, demand=demand2)
                bump_arrival(server2)
            else:
                # COMPLETATO
                recs = sorted(jobs_info[jid]["stage_recs"], key=lambda r: r["pos"])
                row = {
                    "OrigID": jid,
                    "Arrival0": jobs_info[jid]["arrival0"],
                    "Completion": t,
                    "TempoRispostaTotal": t - jobs_info[jid]["arrival0"],
                    "TempoServizioTotaleSeIlJObFosseSOlo": job["demand_total"]
                }
                for i, rec in enumerate(recs, 1):
                    row.update({
                        f"Server_{i}": rec["server"],
                        f"ID_{i}": rec["id_label"],
                        f"IN_{i}": rec["IN"],
                        f"OUT_{i}": rec["OUT"],
                        f"Demand_{i}": rec["Demand"],
                        f"Wait_{i}": rec["Wait"],
                        f"Response_TIme_{i}": rec["Service"],
                        f"N_at_Admit_{i}": rec["N_at_Admit"],
                        f"AvgN_during_{i}": rec["AvgN_during"],
                    })
                jobs_rows.append(row)

    # ===== output base =====
    K=5
    base_cols=["OrigID","Arrival0","Completion","TempoRispostaTotal","TempoServizioTotaleSeIlJObFosseSOlo"]
    per_stage=[]
    for i in range(1,K+1):
        per_stage += [f"Server_{i}", f"ID_{i}", f"IN_{i}", f"OUT_{i}",
                      f"Demand_{i}", f"Wait_{i}", f"Response_TIme_{i}",
                      f"N_at_Admit_{i}", f"AvgN_during_{i}"]
    df = pd.DataFrame(jobs_rows)
    for c in base_cols + per_stage:
        if c not in df.columns: df[c]=pd.NA
    df = df[base_cols+per_stage].sort_values("OrigID").reset_index(drop=True)

    t_end = t if t>0 else 1.0

    # ===== metriche aggiuntive =====
    loss_pct_by_server = {}
    for s in sorted(set(list(arrivals_by_server.keys()) + list(losses_by_server.keys()))):
        a = float(arrivals_by_server.get(s, 0))
        l = float(losses_by_server.get(s, 0))
        loss_pct_by_server[s] = (100.0*l/a) if a > 0 else 0.0

    stage_df = pd.DataFrame(all_stage_records) if all_stage_records else pd.DataFrame(
        columns=["server","pos","Service","AvgN_during"]
    )
    rt_mean_per_visit = (
        stage_df.groupby(["server","pos"])["Service"].mean().sort_index().to_dict()
        if not stage_df.empty else {}
    )
    rt_mean_per_server = (
        stage_df.groupby(["server"])["Service"].mean().sort_index().to_dict()
        if not stage_df.empty else {}
    )
    avgN_per_visit = (
        stage_df.groupby(["server","pos"])["AvgN_during"].mean().sort_index().to_dict()
        if not stage_df.empty else {}
    )
    avgN_per_server_cont = {k: (servers[k].N_time_accum / t_end) for k in servers.keys()}

    stats = {
        "T_end": t_end,
        "AvgN_A": servers["A"].N_time_accum / t_end,
        "AvgN_B": servers["B"].N_time_accum / t_end,
        "AvgN_P": servers["P"].N_time_accum / t_end,
        "utilization": {k: servers[k].busy_time / t_end for k in servers.keys()},
        "arrivals_by_server": arrivals_by_server,
        "losses_by_server":   losses_by_server,
        "loss_pct_by_server": loss_pct_by_server,
        "lost_total": lost_total,
        "arrivals": total_arrivals,
        "pct_lost_total": (100.0*lost_total/total_arrivals) if total_arrivals>0 else 0.0,
        "lost_on_A2": loss_on_a2, "lost_on_B": loss_on_b, "lost_on_P": loss_on_p,
        "rt_mean_per_visit": rt_mean_per_visit,
        "rt_mean_per_server": rt_mean_per_server,
        "avgN_per_visit": avgN_per_visit,
        "avgN_per_server": avgN_per_server_cont
    }
    if "A2" in servers: stats["AvgN_A2"] = servers["A2"].N_time_accum / t_end
    if "B2" in servers: stats["AvgN_B2"] = servers["B2"].N_time_accum / t_end

    # ===== stampa sintetica =====
    print("\n=== METRICHE DI PERDITA ===")
    print(f"Arrivi totali: {total_arrivals}  |  Persi totali: {lost_total}  |  % persi: {stats['pct_lost_total']:.2f}%")
    for s in sorted(loss_pct_by_server.keys()):
        print(f"  - {s}: arrivi={arrivals_by_server.get(s,0)}, perdite={losses_by_server.get(s,0)}, "
              f"%persi={loss_pct_by_server[s]:.2f}%")

    print("\n=== TEMPO DI RISPOSTA MEDIO (Service) PER VISITA ===")
    if rt_mean_per_visit:
        for (srv,pos), val in rt_mean_per_visit.items():
            print(f"  - server={srv} pos={pos}: RTmedio={val:.6f}")
    else:
        print("  (nessuna visita registrata)")

    print("\n=== TEMPO DI RISPOSTA MEDIO (Service) PER SERVER ===")
    if rt_mean_per_server:
        for srv, val in rt_mean_per_server.items():
            print(f"  - server={srv}: RTmedio={val:.6f}")
    else:
        print("  (nessuna visita registrata)")

    print("\n=== N MEDIO (occupazione) ===")
    print("Per server (continuo, ∫N(t)dt/T):")
    for srv, val in avgN_per_server_cont.items():
        print(f"  - {srv}: AvgN={val:.6f}")
    print("Per visita (media AvgN_during):")
    if avgN_per_visit:
        for (srv,pos), val in avgN_per_visit.items():
            print(f"  - server={srv} pos={pos}: AvgN_visit={val:.6f}")
    else:
        print("  (nessuna visita registrata)")

    return df, stats


In [8]:

df_jobs, stats = simulate_ps_ABAPA_scaled_AB_with_spillover_and_loss_tags(
    lmb=0.5,
    mu_A1=5.0, mu_B=1.25, mu_A2=2.5, mu_P=2.5, mu_A3=10.0,
    threshold_time=800000,
    k_threshold_A_up=0.6, k_threshold_A_down=0.3,
    k_threshold_B_up=0.6, k_threshold_B_down=0.3,
)

print(stats)
print(df_jobs.head(2000).to_string(index=False))




# SImulazione con LMB variabile

In [27]:
import math
import heapq
import pandas as pd

def simulate_ps_ABAPA_scaled_AB_with_spillover_and_loss_tags_lmdVariabile(
    lmb,   # ignorato, usiamo λ(t) a fasce orarie
    mu_A1, mu_B, mu_A2, mu_P, mu_A3,
    threshold_time,
    *,
    # thresholds per scale-out/in
    k_threshold_A_up=0.5, k_threshold_A_down=0.2,
    k_threshold_B_up=0.5, k_threshold_B_down=0.2,
    # probing/adattamento su P
    p_send_to_P_init=0.6, p_probe=0.45, window_size=6, L_target=8.0, gamma=0.25,
    # domanda fissa usata in spillover
    spillover_demand_const=0.6,
    eps=1e-12
):
    """
    Versione PS ABAPA con:
      - Scaling A/A2 e B/B2
      - Spillover su A nella terza visita
      - Perdita con tag e FLAG
      - Arrivi con λ(t) variabile per SABATO e DOMENICA
        in base alle fasce orarie fornite.

    NOTA: lmb è ignorato, la domanda è guidata da λ(t) a fasce orarie.
    """

    # ===== helpers RNG su stream =====
    def U01_from(stream_id):
        selectStream(stream_id)
        u = random()
        if u <= 0.0: return 1e-15
        if u >= 1.0: return 1.0 - 1e-15
        return u

    # ===== server PS tag-based =====
    class PSServer:
        __slots__ = ("name", "c", "S", "n", "heap", "N_time_accum", "busy_time")
        def __init__(self, name, c=1.0):
            self.name = name
            self.c = c
            self.S = 0.0
            self.n = 0
            self.heap = []                 # (F, jid, seg)
            self.N_time_accum = 0.0        # ∫ N(t) dt
            self.busy_time = 0.0           # ∫ 1{n>0} dt

    # server base
    servers = {k: PSServer(k, 1.0) for k in ("A", "B", "P")}

    # === scaling & routing dinamico (A/A2, B/B2) ===
    def choose_A_server(now):
        util_A  = servers["A"].busy_time / now if now > 0 else 0.0
        util_A2 = servers["A2"].busy_time / now if ("A2" in servers and now > 0) else 0.0
        if util_A > k_threshold_A_up:
            if "A2" not in servers:
                servers["A2"] = PSServer("A2", 1.0)
            invA  = 1.0/util_A  if util_A  > 0 else 1.0
            invA2 = 1.0/util_A2 if util_A2 > 0 else 1.0
            p_A = invA / (invA + invA2)
            return "A" if U01_from(STREAM_LB_A) < p_A else "A2"
        return "A"

    def choose_B_server(now):
        util_B  = servers["B"].busy_time / now if now > 0 else 0.0
        util_B2 = servers["B2"].busy_time / now if ("B2" in servers and now > 0) else 0.0
        if util_B > k_threshold_B_up:
            if "B2" not in servers:
                servers["B2"] = PSServer("B2", 1.0)
            invB  = 1.0/util_B  if util_B  > 0 else 1.0
            invB2 = 1.0/util_B2 if util_B2 > 0 else 1.0
            p_B = invB / (invB + invB2)
            return "B" if U01_from(STREAM_LB_B) < p_B else "B2"
        return "B"

    def stage_server_name(pos, now):
        # Routing logico: pos in {1,2,3,4,5} -> A/B/A/P/A (+ eventuali A2/B2)
        if pos in (1, 3, 5):
            return choose_A_server(now)
        if pos == 2:
            return choose_B_server(now)
        return "P"

    def check_scaledown(now):
        if now <= 0:
            return
        util_A = servers["A"].busy_time / now
        if "A2" in servers and servers["A2"].n == 0 and util_A < k_threshold_A_down:
            del servers["A2"]
        util_B = servers["B"].busy_time / now
        if "B2" in servers and servers["B2"].n == 0 and util_B < k_threshold_B_down:
            del servers["B2"]

    # === sequenza visite & indice visita in A ===
    def a_visit_index_after(pos):
        # pos: 1(A1), 2(B), 3(A2), 4(P), 5(A3)
        # etichettatura delle visite in A: 1,1,2,2,3
        return [1, 1, 2, 2, 3][pos-1]

    # === Domanda di servizio, stream separati A/A2, B/B2 ===
    def draw_demand_for(pos, server_name):
        """
        Estrae la domanda di servizio per la visita `pos` e server fisico `server_name`,
        usando gli stream separati A/A2 e B/B2.
        """
        # 1ª visita in A
        if pos == 1:
            if server_name == "A":
                selectStream(STREAM_A1_A)
            else:
                selectStream(STREAM_A1_A2)
            return Exponential(1.0 / mu_A1)

        # B
        elif pos == 2:
            if server_name == "B":
                selectStream(STREAM_B_1)
            else:
                selectStream(STREAM_B_2)
            return Exponential(1.0 / mu_B)

        # 2ª visita in A
        elif pos == 3:
            if server_name == "A":
                selectStream(STREAM_A2_A)
            else:
                selectStream(STREAM_A2_A2)
            return Exponential(1.0 / mu_A2)

        # P
        elif pos == 4:
            selectStream(STREAM_P)
            return Exponential(1.0 / mu_P)

        # 3ª visita in A
        else:  # pos == 5
            if server_name == "A":
                selectStream(STREAM_A3_A)
            else:
                selectStream(STREAM_A3_A2)
            return Exponential(1.0 / mu_A3)

    # ===== ARRIVI con λ(t) a fasce orarie (Sabato/Domenica) =====
        # ===== ARRIVI con λ(t) a fasce orarie (Sabato/Domenica) =====
    # t è in SECONDI
    # t in [0, 86400)   -> Sabato
    # t in [86400, 2*86400) -> Domenica

       # ===== ARRIVI con λ(t) a fasce orarie (Sabato/Domenica), t in SECONDI =====
    # t in [0, 86400)        -> Sabato
    # t in [86400, 2*86400) -> Domenica
    # Oltre questi 2 giorni: niente arrivi (λ(t) = 0).

    # Fasce orarie (in ore) e λ in 1/secondo:
    # 00-06, 06-09, 09-12, 12-17, 17-21, 21-24
    LAMBDA_WEEKEND = {
        0: [  # Sabato  (day_idx = 0)
            1.176,  # 00-06
            1.224,  # 06-09
            1.230,  # 09-12
            1.351,  # 12-17
            1.251,  # 17-21
            1.126   # 21-24
        ],
        1: [  # Domenica (day_idx = 1)
            1.130,  # 00-06
            1.240,  # 06-09
            1.287,  # 09-12
            1.379,  # 12-17
            1.244,  # 17-21
            1.166   # 21-24
        ],
    }

    # Bound superiore per thinning (già in 1/secondo)
    LAMBDA_MAX = max(
        max(LAMBDA_WEEKEND[0]),
        max(LAMBDA_WEEKEND[1])
    )

    def lambda_t(now):
        """
        λ(t) piecewise-costante in funzione dell'ora del giorno e del giorno (Sab/Dom).
        t è in SECONDI.
        Se now >= threshold_time -> 0 (nessun arrivo).
        Se day_idx >= 2 -> 0 (oltre Domenica).
        """
        if now >= threshold_time:
            return 0.0

        # giorno = floor(t / 86400)
        day_idx = int(now // 86400)
        if day_idx < 0:
            day_idx = 0
        if day_idx >= 2:
            return 0.0  # oltre sab+dom, nessun arrivo

        # ora del giorno in [0,24)
        hour_in_day = (now - 86400.0 * day_idx) / 3600.0

        # individuazione fascia oraria
        if   hour_in_day < 6.0:  slot = 0   # 00-06
        elif hour_in_day < 9.0:  slot = 1   # 06-09
        elif hour_in_day < 12.0: slot = 2   # 09-12
        elif hour_in_day < 17.0: slot = 3   # 12-17
        elif hour_in_day < 21.0: slot = 4   # 17-21
        else:                    slot = 5   # 21-24

        # λ già in 1/secondo
        return LAMBDA_WEEKEND[day_idx][slot]

    def schedule_next_arrival(current_time):
        """
        Thinning con bound LAMBDA_MAX (1/sec) usando STREAM_ARR.
        Genera una sequenza di arrivi NHPP con rate λ(t) weekend a fasce.
        """
        t_candidate = current_time
        while True:
            # tempo inter-arrivo candidato con λ* = LAMBDA_MAX
            selectStream(STREAM_ARR)
            w = Exponential(1.0 / LAMBDA_MAX)  # Exp(rate=LAMBDA_MAX)
            t_candidate = t_candidate + w
            if t_candidate > threshold_time:
                return None

            # accetta con prob λ(t) / LAMBDA_MAX
            selectStream(STREAM_ARR)
            u = random()
            lam = lambda_t(t_candidate)
            ratio = lam / LAMBDA_MAX if LAMBDA_MAX > 0 else 0.0
            if u <= (ratio + 1e-15):
                return t_candidate
            # altrimenti rifiuta e riprova



    # ===== stato globale =====
    t = 0.0
    next_arrival = schedule_next_arrival(t)  # primo arrivo

    job_id = 0
    jobs_rows = []
    jobs_info = {}
    all_stage_records = []

    p_send_to_P = float(p_send_to_P_init)
    obs_lat_P = []

    arrivals_by_server, losses_by_server = {}, {}
    lost_total = loss_on_b = loss_on_p = loss_on_a2 = 0
    total_arrivals = 0

    def bump_arrival(s):
        arrivals_by_server[s] = arrivals_by_server.get(s, 0) + 1

    def bump_loss(s):
        losses_by_server[s] = losses_by_server.get(s, 0) + 1

    # === advance ===
    def advance(delta):
        nonlocal t
        if delta <= 0:
            return
        for srv in servers.values():
            srv.N_time_accum += srv.n * delta
            if srv.n > 0:
                srv.busy_time += delta
        for srv in servers.values():
            if srv.n > 0:
                srv.S += (srv.c / srv.n) * delta
        t += delta

    # ammissione segmento
    def admit_segment(jid, pos, server_name, arrival_t, demand=None):
        srv = servers[server_name]
        if demand is None:
            d = draw_demand_for(pos, server_name)
        else:
            d = demand
        n_at_admit = srv.n + 1
        seg = {
            "job_id": jid,
            "pos": pos,
            "server": server_name,
            "a_visit": a_visit_index_after(pos),
            "arrival_stage": arrival_t,
            "IN": arrival_t,
            "S_admit": srv.S,
            "Demand": d,
            "N_at_Admit": n_at_admit
        }
        F = seg["S_admit"] + d
        heapq.heappush(srv.heap, (F, jid, seg))
        srv.n += 1
        return seg

    INF = math.inf

    def arrivals_open():
        return (next_arrival is not None and next_arrival <= threshold_time)

    def work_left():
        return arrivals_open() or any(srv.n > 0 for srv in servers.values())

    # ===== MAIN LOOP =====
    while work_left():
        # prossimo completion (regola tag-based)
        next_comp_time = INF
        comp_srv = None
        for srv in servers.values():
            if srv.n == 0 or not srv.heap:
                continue
            F_min, _, _ = srv.heap[0]
            if F_min <= srv.S + 1e-15:
                dt_srv = 0.0
            else:
                dS = F_min - srv.S
                dt_srv = dS * (srv.n / srv.c)
            tc = t + dt_srv
            if tc < next_comp_time:
                next_comp_time = tc
                comp_srv = srv

        ta = next_arrival if arrivals_open() else INF
        t_next = min(ta, next_comp_time)
        if math.isinf(t_next):
            break

        advance(t_next - t)
        check_scaledown(t)

        # === ARRIVO ===
        if abs(t_next - ta) <= 1e-12 and arrivals_open():
            job_id += 1
            jid = job_id
            total_arrivals += 1
            jobs_info[jid] = {
                "arrival0": t,
                "stage_recs": [],
                "demand_total": 0.0,
                "spillover_used": False,
                "is_probe": (U01_from(STREAM_PROBE) < p_probe)
            }

            pos = 1
            server_name = stage_server_name(pos, t)
            admit_segment(jid, pos, server_name, t)
            bump_arrival(server_name)

            # pianifica il prossimo arrivo
            next_arrival = schedule_next_arrival(t)

        # === COMPLETION ===
        else:
            if comp_srv is None:
                break  # sicurezza

            F_min, jid, seg = heapq.heappop(comp_srv.heap)
            comp_srv.n -= 1

            IN = seg["IN"]
            OUT = t
            Demand = seg["Demand"]
            Wait = 0.0  # PS puro senza attesa dedicata
            Service = OUT - IN
            n_at_admit = seg["N_at_Admit"]
            avgN_during = n_at_admit

            rec = {
                "pos": seg["pos"],
                "server": seg["server"],
                "id_label": f"{jid}[{seg['a_visit']}]",
                "IN": IN,
                "OUT": OUT,
                "Demand": Demand,
                "Wait": Wait,
                "Service": Service,
                "N_at_Admit": n_at_admit,
                "AvgN_during": avgN_during
            }
            job = jobs_info[jid]
            job["stage_recs"].append(rec)
            all_stage_records.append(rec)
            job["demand_total"] += Demand

            # feedback su P (solo probe)
            if seg["server"] == "P" and job.get("is_probe", False):
                obs_lat_P.append(Service + Wait)
                if len(obs_lat_P) > window_size:
                    obs_lat_P.pop(0)
                if len(obs_lat_P) >= window_size:
                    Lhat = sum(obs_lat_P) / len(obs_lat_P)
                    if Lhat > L_target:
                        p_send_to_P = max(0.0, p_send_to_P * (1 - gamma))
                    else:
                        p_send_to_P = min(1.0, p_send_to_P * (1 + gamma))

            # === perdite ai confini ===
            lost_here = False
            if FLAG:
                if seg["pos"] == 2:
                    p_loss = float(loss_probability_from_sojourn_B(Service))
                    lost_here = (U01_from(STREAM_LOSS_B) < p_loss)
                    if lost_here:
                        loss_on_b += 1
                elif seg["pos"] == 3 and not job["spillover_used"]:
                    p_loss = 0.1
                    lost_here = (U01_from(STREAM_LOSS_A2) < p_loss)
                    if lost_here:
                        loss_on_a2 += 1
                elif seg["pos"] == 4:
                    p_loss = float(loss_prob_class2_after_P(Service))
                    lost_here = (U01_from(STREAM_LOSS_P) < p_loss)
                    if lost_here:
                        loss_on_p += 1
                elif seg["pos"] == 5:
                    p_loss = float(loss_prob_class3_at_exit(Service))
                    lost_here = (U01_from(STREAM_LOSS_EXIT) < p_loss)

            if lost_here:
                lost_total += 1
                bump_loss(seg["server"])
                continue

            # routing o fine job
            if seg["pos"] < 5:
                pos2 = seg["pos"] + 1

                if pos2 == 4:
                    # decide spillover vs P con lo stream delle probe
                    do_spill = (U01_from(STREAM_PROBE) > p_send_to_P)
                    if do_spill and (not job["spillover_used"]):
                        # spillover: rientro in A/A2 con domanda fissa
                        server2 = stage_server_name(3, t)   # A/A2
                        demand2 = spillover_demand_const
                        pos2 = seg["pos"]                   # resta nello stage numerico (3)
                        job["spillover_used"] = True
                    else:
                        server2 = "P"
                        demand2 = draw_demand_for(4, "P")
                        pos2 = 4
                else:
                    server2 = stage_server_name(pos2, t)
                    demand2 = draw_demand_for(pos2, server2)

                admit_segment(jid, pos2, server2, t, demand=demand2)
                bump_arrival(server2)

            else:
                # COMPLETATO (pos == 5)
                recs = sorted(jobs_info[jid]["stage_recs"], key=lambda r: r["pos"])
                row = {
                    "OrigID": jid,
                    "Arrival0": jobs_info[jid]["arrival0"],
                    "Completion": t,
                    "TempoRispostaTotal": t - jobs_info[jid]["arrival0"],
                    "TempoServizioTotaleSeIlJObFosseSOlo": job["demand_total"]
                }
                for i, r in enumerate(recs, 1):
                    row.update({
                        f"Server_{i}": r["server"],
                        f"ID_{i}": r["id_label"],
                        f"IN_{i}": r["IN"],
                        f"OUT_{i}": r["OUT"],
                        f"Demand_{i}": r["Demand"],
                        f"Wait_{i}": r["Wait"],
                        f"Response_TIme_{i}": r["Service"],
                        f"N_at_Admit_{i}": r["N_at_Admit"],
                        f"AvgN_during_{i}": r["AvgN_during"],
                    })
                jobs_rows.append(row)

    # ===== costruzione DataFrame =====
    K = 5
    base_cols = [
        "OrigID", "Arrival0", "Completion",
        "TempoRispostaTotal", "TempoServizioTotaleSeIlJObFosseSOlo"
    ]
    per_stage = []
    for i in range(1, K+1):
        per_stage += [
            f"Server_{i}", f"ID_{i}", f"IN_{i}", f"OUT_{i}",
            f"Demand_{i}", f"Wait_{i}", f"Response_TIme_{i}",
            f"N_at_Admit_{i}", f"AvgN_during_{i}"
        ]
    df = pd.DataFrame(jobs_rows)
    for c in base_cols + per_stage:
        if c not in df.columns:
            df[c] = pd.NA
    df = df[base_cols + per_stage].sort_values("OrigID").reset_index(drop=True)

    t_end = t if t > 0 else 1.0

    # ===== metriche aggiuntive =====
    loss_pct_by_server = {}
    for s in sorted(set(list(arrivals_by_server.keys()) + list(losses_by_server.keys()))):
        a = float(arrivals_by_server.get(s, 0))
        l = float(losses_by_server.get(s, 0))
        loss_pct_by_server[s] = (100.0 * l / a) if a > 0 else 0.0

    stage_df = pd.DataFrame(all_stage_records) if all_stage_records else pd.DataFrame(
        columns=["server", "pos", "Service", "AvgN_during"]
    )
    rt_mean_per_visit = (
        stage_df.groupby(["server", "pos"])["Service"].mean().sort_index().to_dict()
        if not stage_df.empty else {}
    )
    rt_mean_per_server = (
        stage_df.groupby(["server"])["Service"].mean().sort_index().to_dict()
        if not stage_df.empty else {}
    )
    avgN_per_visit = (
        stage_df.groupby(["server", "pos"])["AvgN_during"].mean().sort_index().to_dict()
        if not stage_df.empty else {}
    )
    avgN_per_server_cont = {k: (servers[k].N_time_accum / t_end) for k in servers.keys()}

    stats = {
        "T_end": t_end,
        "AvgN_A": servers["A"].N_time_accum / t_end,
        "AvgN_B": servers["B"].N_time_accum / t_end,
        "AvgN_P": servers["P"].N_time_accum / t_end,
        "utilization": {k: servers[k].busy_time / t_end for k in servers.keys()},
        "arrivals_by_server": arrivals_by_server,
        "losses_by_server":   losses_by_server,
        "loss_pct_by_server": loss_pct_by_server,
        "lost_total": lost_total,
        "arrivals": total_arrivals,
        "pct_lost_total": (100.0 * lost_total / total_arrivals) if total_arrivals > 0 else 0.0,
        "lost_on_A2": loss_on_a2,
        "lost_on_B": loss_on_b,
        "lost_on_P": loss_on_p,
        "rt_mean_per_visit": rt_mean_per_visit,
        "rt_mean_per_server": rt_mean_per_server,
        "avgN_per_visit": avgN_per_visit,
        "avgN_per_server": avgN_per_server_cont
    }
    if "A2" in servers:
        stats["AvgN_A2"] = servers["A2"].N_time_accum / t_end
    if "B2" in servers:
        stats["AvgN_B2"] = servers["B2"].N_time_accum / t_end

    # ===== stampa sintetica =====
    print("\n=== METRICHE DI PERDITA ===")
    print(f"Arrivi totali: {total_arrivals}  |  Persi totali: {lost_total}  |  % persi: {stats['pct_lost_total']:.2f}%")
    for s in sorted(loss_pct_by_server.keys()):
        print(f"  - {s}: arrivi={arrivals_by_server.get(s,0)}, perdite={losses_by_server.get(s,0)}, "
              f"%persi={loss_pct_by_server[s]:.2f}%")

    print("\n=== TEMPO DI RISPOSTA MEDIO (Service) PER VISITA ===")
    if rt_mean_per_visit:
        for (srv, pos), val in rt_mean_per_visit.items():
            print(f"  - server={srv} pos={pos}: RTmedio={val:.6f}")
    else:
        print("  (nessuna visita registrata)")

    print("\n=== TEMPO DI RISPOSTA MEDIO (Service) PER SERVER ===")
    if rt_mean_per_server:
        for srv, val in rt_mean_per_server.items():
            print(f"  - server={srv}: RTmedio={val:.6f}")
    else:
        print("  (nessuna visita registrata)")

    print("\n=== N MEDIO (occupazione) ===")
    print("Per server (continuo, ∫N(t)dt/T):")
    for srv, val in avgN_per_server_cont.items():
        print(f"  - {srv}: AvgN={val:.6f}")
    print("Per visita (media AvgN_during):")
    if avgN_per_visit:
        for (srv, pos), val in avgN_per_visit.items():
            print(f"  - server={srv} pos={pos}: AvgN_visit={val:.6f}")
    else:
        print("  (nessuna visita registrata)")

    return df, stats


In [ ]:

# Richiamo (nota: lmb viene ignorato in modalità NHPP)
df_run, stats = simulate_ps_ABAPA_scaled_AB_with_spillover_and_loss_tags_lmdVariabile(
    lmb=0.5,
    mu_A1=5.0, mu_B=1.25, mu_A2=2.5, mu_P=1.42, mu_A3=6.67,
    threshold_time=1728000,
    # opzionali (lasciati come default o personalizza):
    k_threshold_A_up=0.5, k_threshold_A_down=0.2,
    k_threshold_B_up=0.5, k_threshold_B_down=0.2,
    p_send_to_P_init=0.6, p_probe=0.45, window_size=6, L_target=8.0, gamma=0.25,
    spillover_demand_const=0.6
)

In [31]:
import os
from pathlib import Path
import gc
import pandas as pd

dfs_seeds = {}

plantSeeds(12348948)

num_runs = 15

out_dir = Path("PMCSN/SimulazioniModelloMigliorativo")
out_dir.mkdir(parents=True, exist_ok=True)

# percorso del file unico con le stats di tutte le run
stats_path = out_dir / "stats_all_runs.csv"

for i in range(num_runs):
    print(f"--- RUN {i} seeds PRIMA ---")
    _ = print_all_streamsMigliorativo()
    dfs_seeds[i] = snapshot_streamsMigliorativo()
    
    df_jobs, stats = simulate_ps_ABAPA_scaled_AB_with_spillover_and_loss_tags_lmdVariabile(
        lmb=0.5,  # ignorato
        mu_A1=5.0, mu_B=1.25, mu_A2=2.5, mu_P=2.5, mu_A3=10.0,
        threshold_time=86400*2,
        k_threshold_A_up=0.5, k_threshold_A_down=0.2,
        k_threshold_B_up=0.5, k_threshold_B_down=0.2,
        p_send_to_P_init=0.6, p_probe=0.45,
        window_size=6, L_target=8.0, gamma=0.25,
        spillover_demand_const=0.6
    )

    # ======= SCRIVI SUBITO SU DISCO =======
    # 1) JOB COMPLETATI
    df_jobs.to_csv(out_dir / f"jobs_run_{i:03d}.csv", index=False)

    # 2) STATS FLATTEN -> UNA RIGA PER RUN
    row_stats = {
        "run": i,
        "T_end": stats.get("T_end", float("nan")),
        "arrivals": stats.get("arrivals", float("nan")),
        "lost_total": stats.get("lost_total", float("nan")),
        "pct_lost_total": stats.get("pct_lost_total", float("nan")),
    }

    # utilizzo per server: util_A, util_B, util_P, util_A2, util_B2 ...
    for srv, rho in stats.get("utilization", {}).items():
        row_stats[f"util_{srv}"] = rho

    # perdite % per server: loss_pct_A, loss_pct_B, loss_pct_P, ...
    for srv, pct in stats.get("loss_pct_by_server", {}).items():
        row_stats[f"loss_pct_{srv}"] = pct

    # eventualmente: arrivi_per_server, perdite_per_server
    for srv, a in stats.get("arrivals_by_server", {}).items():
        row_stats[f"arrivals_{srv}"] = a
    for srv, l in stats.get("losses_by_server", {}).items():
        row_stats[f"losses_{srv}"] = l

    stats_df = pd.DataFrame([row_stats])

    # append su file unico stats_all_runs.csv
    if stats_path.exists():
        stats_df.to_csv(stats_path, mode="a", header=False, index=False)
    else:
        stats_df.to_csv(stats_path, mode="w", header=True, index=False)

    # pulizia memoria
    del df_jobs, stats, stats_df
    gc.collect()
    # =====================================

    print(f"--- RUN {i} seeds DOPO ---")
    _ = print_all_streamsMigliorativo()

# seeds tutti insieme
write_streams_csvMigliorativo(
    "PMCSN/seedOrizzonteFinitoMigliorativo2Fact.csv",
    dfs_seeds,
    run_field="RUN",
    encoding="utf-8"
)


# Transitorio modello migliorativo

In [60]:



dfs_jobs = {}
dfs_seeds = {}

plantSeeds(12348948)

num_runs = 5
for i in range(num_runs):
    print(f"--- RUN {i} seeds PRIMA ---")
    _ = print_all_streamsMigliorativo() 
    dfs_seeds[i] = snapshot_streamsMigliorativo()    

    df_jobs, stats = simulate_ps_ABAPA_scaled_AB_with_spillover_and_loss_tags(
    lmb=0.5,
    mu_A1=5.0, mu_B=1.25, mu_A2=2.5, mu_P=2.5, mu_A3=10.0,
    threshold_time=1000000,
    k_threshold_A_up=0.6, k_threshold_A_down=0.3,
    k_threshold_B_up=0.6, k_threshold_B_down=0.3,
    )

    dfs_jobs[i] = df_jobs

    print(f"--- RUN {i} seeds DOPO ---")
    _ = print_all_streamsMigliorativo()               


write_streams_csvMigliorativo("PMCSN/seedMigliorativo.csv", dfs_seeds, run_field="RUN", encoding="utf-8")


In [70]:
run_seeds = [
    12348948,
    1643109367,
    731224787,
    1878977007,
    2094133940
]

fig, ax, thr_stats = plot_throughput_per_scenario(
    dfs_jobs,
    run_seeds=run_seeds,
    x_col="Completion",
    bin_size=800,          # stesso ordine che stavi usando
    x_max=300000,
    ema_alpha=0.01,        # leggero smoothing per visual
    per_unit="per_second", # oppure "per_minute"/"per_hour"
    title="Throughput (completamenti/bin)"
)

fig, ax = plot_utilization_per_scenario(
    dfs_jobs,
    run_seeds=run_seeds,
    servers=("A","B","P"),
    bin_size=800,
    x_max=200000,
    ema_alpha=0.02,   
    title="Utilizzazione per scenario e server"
)

fig, ax, _ = plot_rt_mean_per_scenario(
    dfs_jobs, x_col="Completion", y_col="TempoRispostaTotal",
    run_seeds=run_seeds,
    bin_size=800, x_max=200000, ema_alpha=0.02,y_pad_frac=1
  
)

fig, ax, N_stats = plot_avgN_system_per_run(
    dfs_jobs,
    run_seeds=run_seeds,
    bin_size=800.0,
    normalize_time=False,
    ema_alpha=0.02,
    x_max=171000  
)

# Grafici validazione di entrambi i sistemi al variare di lmb

In [12]:


def sweep_and_plot_ABAPA(
    mode: str = "scaled",                 # "fast" oppure "scaled"
    lmb_start: float = 0.5,
    lmb_end: float = 1.4,
    lmb_step: float = 0.1,
    *,
    # parametri comuni
    mu_A1: float = 5.0,
    mu_B: float = 1.25,
    mu_A2: float = 2.5,
    mu_P: float = 2.5,
    mu_A3: float = 10.0,
    threshold_time: float = 1_000_000.0,
    # kwargs extra passati SOLO alla simulazione "scaled"
    scaled_kwargs: dict | None = None
) -> pd.DataFrame:
    """
    Esegue uno sweep su λ e disegna:
      - Tempo di risposta medio totale vs λ
      - Numero medio di job nel sistema vs λ
      - Utilizzazione per server vs λ (supporto dinamico: A, B, P e anche A2/B2 se presenti)
      - [solo mode='scaled'] Job persi per server vs λ

    Ritorna un DataFrame con una riga per λ e colonne con metriche aggregate.
    """
    if mode not in {"fast", "scaled"}:
        raise ValueError("mode deve essere 'fast' oppure 'scaled'.")

    scaled_kwargs = scaled_kwargs or {}
    results = []

    # generazione λ includendo l'estremo destro
    lambdas = np.arange(lmb_start, lmb_end + 1e-12, lmb_step)
    lambdas = np.round(lambdas, 10)

    # per gestire colonne dinamiche (utilizzazioni/loss) collezioniamo le chiavi viste
    all_util_keys: set[str] = set()
    all_loss_keys: set[str] = set()

    for lmb in lambdas:
        if mode == "fast":
            df, stats = simulate_ps_ABAPA_fast(
                lmb=lmb, mu_A1=mu_A1, mu_B=mu_B, mu_A2=mu_A2, mu_P=mu_P, mu_A3=mu_A3,
                threshold_time=threshold_time
            )
            # util dai "work" (Demand) diviso T_end (metodo robusto anche per fast)
            # ma in fast prima avevamo ricavato util con i Demand; qui preferiamo,
            # quando disponibili, usare direttamente AvgN_* e busy_time solo se forniti.
            # Tuttavia la fast non ritorna busy_time/utilization: quindi calcoliamo come prima:
            # Util per A, B, P via somma dei Demand_i per server / T_end
            T_end = float(stats.get("T_end", float("nan")))
            util_dict = {}
            if not df.empty and np.isfinite(T_end) and T_end > 0:
                total_work = {"A": 0.0, "B": 0.0, "P": 0.0}
                stage_pairs = [(f"Server_{i}", f"Demand_{i}") for i in range(1, 6)]
                for s_col, d_col in stage_pairs:
                    if s_col in df.columns and d_col in df.columns:
                        mask = df[s_col].notna() & df[d_col].notna()
                        if mask.any():
                            sub = df.loc[mask, [s_col, d_col]]
                            for name in ("A", "B", "P"):
                                m = sub[s_col] == name
                                if m.any():
                                    total_work[name] += float(sub.loc[m, d_col].sum())
                util_dict = {k: (v / T_end) for k, v in total_work.items()}
            else:
                util_dict = {"A": np.nan, "B": np.nan, "P": np.nan}

            # N medio totale (somma AvgN_* noti)
            avgN_total = float(
                stats.get("AvgN_A", 0.0) + stats.get("AvgN_B", 0.0) + stats.get("AvgN_P", 0.0)
            )

            losses_by_server = {}  # la fast non ha perdite

        else:
            # === modalità scaled ===
            df, stats = simulate_ps_ABAPA_scaled_AB_with_spillover_and_loss_tags(
                lmb=lmb, mu_A1=mu_A1, mu_B=mu_B, mu_A2=mu_A2, mu_P=mu_P, mu_A3=mu_A3,
                threshold_time=threshold_time,
                **scaled_kwargs
            )
            T_end = float(stats.get("T_end", float("nan")))

            # Utilizzazioni direttamente dal dict che ritorna la scaled
            util_dict = dict(stats.get("utilization", {}))

            # N medio totale dalla somma degli AvgN per server continui (se disponibile),
            # altrimenti fallback su A+B+P+(A2/B2 se presenti)
            if "avgN_per_server" in stats and isinstance(stats["avgN_per_server"], dict):
                avgN_total = float(sum(stats["avgN_per_server"].values()))
            else:
                avgN_total = 0.0
                for key in ("A", "B", "P", "A2", "B2"):
                    if key in stats:
                        val = stats.get(key, np.nan)
                        if np.isfinite(val):
                            avgN_total += float(val)
                # fallback minimo
                if avgN_total == 0.0:
                    avgN_total = float(
                        stats.get("AvgN_A", 0.0) + stats.get("AvgN_B", 0.0) + stats.get("AvgN_P", 0.0) +
                        stats.get("AvgN_A2", 0.0) + stats.get("AvgN_B2", 0.0)
                    )

            # Perdite per server
            losses_by_server = dict(stats.get("losses_by_server", {}))
            all_loss_keys.update(losses_by_server.keys())

        # Tempo medio di risposta totale (sojourn)
        mean_rt = float(df["TempoRispostaTotal"].dropna().mean()) if not df.empty else float("nan")

        # aggiorna set chiavi util
        all_util_keys.update(util_dict.keys())

        # riga risultati
        row = {
            "lambda": float(lmb),
            "TempoRispostaMedio": mean_rt,
            "AvgN_Totale": avgN_total,
            "T_end": float(stats.get("T_end", float("nan")))
        }
        # append utilizzi
        for k, v in util_dict.items():
            row[f"Util_{k}"] = float(v) if v is not None else np.nan
        # append perdite (solo scaled)
        for k in all_loss_keys.union(losses_by_server.keys()):
            row[f"Loss_{k}"] = float(losses_by_server.get(k, 0))

        results.append(row)

    # costruiamo il DataFrame completo con tutte le colonne viste
    res_df = pd.DataFrame(results).sort_values("lambda").reset_index(drop=True)

    # --- Plot 1: Tempo di risposta medio totale vs λ ---
    plt.figure()
    plt.plot(res_df["lambda"], res_df["TempoRispostaMedio"], marker="o")
    plt.xlabel("λ (arrivi/tempo)")
    plt.ylabel("Tempo di risposta medio totale")
    plt.title("Tempo di risposta medio totale vs λ")
    plt.grid(True)
    plt.show()

    # --- Plot 2: Numero medio di job nel sistema vs λ ---
    plt.figure()
    plt.plot(res_df["lambda"], res_df["AvgN_Totale"], marker="o")
    plt.xlabel("λ (arrivi/tempo)")
    plt.ylabel("Numero medio di job nel sistema")
    plt.title("Numero medio di job nel sistema vs λ")
    plt.grid(True)
    plt.show()

    # --- Plot 3: Utilizzazione per server vs λ (dinamico) ---
    util_cols = [c for c in res_df.columns if c.startswith("Util_")]
    if util_cols:
        plt.figure()
        for c in util_cols:
            plt.plot(res_df["lambda"], res_df[c], marker="o", label=c.replace("Util_", ""))
        plt.xlabel("λ (arrivi/tempo)")
        plt.ylabel("Utilizzazione ρ")
        plt.title("Utilizzazione per server vs λ")
        plt.legend()
        plt.grid(True)
        plt.show()

    # --- Plot 4 (solo scaled): Job persi per server vs λ ---
    if mode == "scaled":
        loss_cols = [c for c in res_df.columns if c.startswith("Loss_")]
        if loss_cols:
            plt.figure()
            for c in loss_cols:
                plt.plot(res_df["lambda"], res_df[c], marker="o", label=c.replace("Loss_", ""))
            plt.xlabel("λ (arrivi/tempo)")
            plt.ylabel("Job persi (conteggio)")
            plt.title("Job persi per server vs λ")
            plt.legend()
            plt.grid(True)
            plt.show()

    return res_df

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def sweep_and_plot_ABAPA_2grafici(
    mode: str = "scaled",                 # "fast" oppure "scaled"
    lmb_start: float = 0.5,
    lmb_end: float = 1.4,
    lmb_step: float = 0.1,
    *,
    # ---- Config 1 (base) ----
    mu_A1: float = 5.0,
    mu_B: float = 1.25,
    mu_A2: float = 2.5,
    mu_P: float = 2.5,
    mu_A3: float = 10.0,
    threshold_time: float = 1_000_000.0,
    scaled_kwargs: dict | None = None,
    # ---- Config 2 (con suffisso "2"); se presenti, attiva il confronto ----
    mu_A12: float | None = None,
    mu_B2: float | None = None,
    mu_A22: float | None = None,
    mu_P2: float | None = None,
    mu_A32: float | None = None,
    scaled_kwargs2: dict | None = None,
    # Etichette per le due configurazioni
    label_config1: str = "Config 1",
    label_config2: str = "Config 2"
) -> pd.DataFrame:
    """
    Esegue uno sweep su λ e produce:
      - Tempo di risposta medio totale vs λ
      - AvgN totale vs λ
      - (se presenti) Utilizzazioni per server vs λ
      - (modalità 'scaled') Perdite per server, Perdite totali e % Perdite vs λ

    Ritorna un DataFrame con tutte le metriche per ciascun λ e configurazione.
    """

    if mode not in {"fast", "scaled"}:
        raise ValueError("mode deve essere 'fast' oppure 'scaled'.")

    scaled_kwargs = scaled_kwargs or {}
    has_cfg2 = any(v is not None for v in [mu_A12, mu_B2, mu_A22, mu_P2, mu_A32]) or (scaled_kwargs2 is not None)
    if scaled_kwargs2 is None:
        scaled_kwargs2 = {}

    lambdas = np.arange(lmb_start, lmb_end + 1e-12, lmb_step)
    lambdas = np.round(lambdas, 10)

    # -------- helper: totale offerto (tentativi) = completati + persi --------
    def _infer_total_offered(stats: dict, losses_total: float, df: pd.DataFrame | None) -> float | None:
        # 1) Preferisci conteggio esplicito di completati
        comp = stats.get("completed_jobs")
        if comp is not None:
            try:
                return float(comp) + float(losses_total)
            except Exception:
                pass

        # 2) Fallback su contatori "attempted" (se includono anche i persi)
        for k in ("attempted_arrivals", "arrivals_attempted", "jobs_generated",
                  "generated_jobs", "total_arrivals"):
            if k in stats and stats[k] is not None:
                try:
                    return float(stats[k])
                except Exception:
                    pass

        # 3) Fallback finale: se il DF contiene UNA riga per job completato,
        #    allora attempted ≈ completati (len(df)) + persi
        if df is not None:
            try:
                return float(len(df)) + float(losses_total)
            except Exception:
                pass

        return None

    # --------------------------- esecuzione dello sweep ---------------------------
    def _run_sweep(cfg_idx: int) -> pd.DataFrame:
        _mu_A1 = mu_A1 if cfg_idx == 1 else (mu_A12 if mu_A12 is not None else mu_A1)
        _mu_B  = mu_B  if cfg_idx == 1 else (mu_B2  if mu_B2  is not None else mu_B)
        _mu_A2 = mu_A2 if cfg_idx == 1 else (mu_A22 if mu_A22 is not None else mu_A2)
        _mu_P  = mu_P  if cfg_idx == 1 else (mu_P2  if mu_P2  is not None else mu_P)
        _mu_A3 = mu_A3 if cfg_idx == 1 else (mu_A32 if mu_A32 is not None else mu_A3)
        _scaled_kwargs = scaled_kwargs if cfg_idx == 1 else scaled_kwargs2

        rows = []
        all_loss_keys: set[str] = set()
        all_util_keys: set[str] = set()

        for lmb in lambdas:
            if mode == "fast":
                df, stats = simulate_ps_ABAPA_fast(
                    lmb=lmb, mu_A1=_mu_A1, mu_B=_mu_B, mu_A2=_mu_A2, mu_P=_mu_P, mu_A3=_mu_A3,
                    threshold_time=threshold_time
                )
                T_end = float(stats.get("T_end", float("nan")))

                # Utilizzazione (ricostruita da DF, solo A/B/P)
                util_dict = {}
                if not df.empty and np.isfinite(T_end) and T_end > 0:
                    total_work = {"A": 0.0, "B": 0.0, "P": 0.0}
                    stage_pairs = [(f"Server_{i}", f"Demand_{i}") for i in range(1, 5+1)]
                    for s_col, d_col in stage_pairs:
                        if s_col in df.columns and d_col in df.columns:
                            mask = df[s_col].notna() & df[d_col].notna()
                            if mask.any():
                                sub = df.loc[mask, [s_col, d_col]]
                                for name in ("A", "B", "P"):
                                    m = sub[s_col] == name
                                    if m.any():
                                        total_work[name] += float(sub.loc[m, d_col].sum())
                    util_dict = {k: (v / T_end) for k, v in total_work.items()}
                else:
                    util_dict = {"A": np.nan, "B": np.nan, "P": np.nan}

                avgN_total = float(
                    stats.get("AvgN_A", 0.0) + stats.get("AvgN_B", 0.0) + stats.get("AvgN_P", 0.0)
                )

                # Nella modalità "fast" non abbiamo perdite per server
                losses_by_server = {}
                losses_total = 0.0
                total_offered = _infer_total_offered(stats, losses_total, df)
                loss_pct = (100.0 * losses_total / total_offered) if (total_offered and total_offered > 0) else np.nan

            else:
                df, stats = simulate_ps_ABAPA_scaled_AB_with_spillover_and_loss_tags(
                    lmb=lmb, mu_A1=_mu_A1, mu_B=_mu_B, mu_A2=_mu_A2, mu_P=_mu_P, mu_A3=_mu_A3,
                    threshold_time=threshold_time,
                    **_scaled_kwargs
                )
                T_end = float(stats.get("T_end", float("nan")))
                util_dict = dict(stats.get("utilization", {}))

                # AvgN totale: prova contatore aggregato se presente
                if "avgN_per_server" in stats and isinstance(stats["avgN_per_server"], dict):
                    avgN_total = float(sum(stats["avgN_per_server"].values()))
                else:
                    # fallback su possibili chiavi legacy
                    avgN_total = 0.0
                    for key in ("A", "B", "P", "A2", "B2", "AvgN_A", "AvgN_B", "AvgN_P", "AvgN_A2", "AvgN_B2"):
                        if key in stats:
                            val = stats.get(key, np.nan)
                            if np.isfinite(val):
                                avgN_total += float(val)

                # Perdite per server (se presenti)
                losses_by_server = dict(stats.get("losses_by_server", {}))
                all_loss_keys.update(losses_by_server.keys())

                losses_total = float(sum(losses_by_server.values())) if losses_by_server else 0.0

                # Denominatore robusto: completed + lost (o contatore attempted), fallback su len(df)
                total_offered = _infer_total_offered(stats, losses_total, df)
                loss_pct = (100.0 * losses_total / total_offered) if (total_offered and total_offered > 0) else np.nan

            # Tempo risposta medio totale
            mean_rt = float(df["TempoRispostaTotal"].dropna().mean()) if isinstance(df, pd.DataFrame) and not df.empty else float("nan")
            all_util_keys.update(util_dict.keys())

            # Riga risultati
            row = {
                "lambda": float(lmb),
                "TempoRispostaMedio": mean_rt,
                "AvgN_Totale": avgN_total,
                "T_end": float(stats.get("T_end", float("nan"))),
                "Loss_Total": losses_total,
                "LossPct_Total": loss_pct,
            }
            for k, v in util_dict.items():
                row[f"Util_{k}"] = float(v) if v is not None else np.nan
            for k in all_loss_keys.union(losses_by_server.keys()):
                row[f"Loss_{k}"] = float(losses_by_server.get(k, 0.0))
            rows.append(row)

        res = pd.DataFrame(rows).sort_values("lambda").reset_index(drop=True)
        return res

    # -------------------------- costruzione risultati & plot --------------------------
    res1 = _run_sweep(cfg_idx=1)
    res1["config"] = label_config1
    if has_cfg2:
        res2 = _run_sweep(cfg_idx=2)
        res2["config"] = label_config2
        res_all = pd.concat([res1, res2], ignore_index=True)
    else:
        res_all = res1.copy()

    # Plot 1: RT medio
    plt.figure()
    for label, df_g in res_all.groupby("config"):
        plt.plot(df_g["lambda"], df_g["TempoRispostaMedio"], marker="o", label=label)
    plt.xlabel("λ (arrivi/tempo)")
    plt.ylabel("Tempo di risposta medio totale")
    plt.title("Tempo di risposta medio totale vs λ")
    plt.grid(True)
    plt.legend()
    plt.show()

    # Plot 2: AvgN totale
    plt.figure()
    for label, df_g in res_all.groupby("config"):
        plt.plot(df_g["lambda"], df_g["AvgN_Totale"], marker="o", label=label)
    plt.xlabel("λ (arrivi/tempo)")
    plt.ylabel("Numero medio di job nel sistema")
    plt.title("Numero medio di job nel sistema vs λ")
    plt.grid(True)
    plt.legend()
    plt.show()

    # Plot 3: Utilizzazioni dinamiche
    util_cols = sorted([c for c in res_all.columns if c.startswith("Util_")])
    if util_cols:
        for c in util_cols:
            plt.figure()
            for label, df_g in res_all.groupby("config"):
                plt.plot(df_g["lambda"], df_g[c], marker="o", label=label)
            plt.xlabel("λ (arrivi/tempo)")
            plt.ylabel(f"Utilizzazione ρ ({c.replace('Util_', '')})")
            plt.title(f"Utilizzazione {c.replace('Util_', '')} vs λ")
            plt.grid(True)
            plt.legend()
            plt.show()

    # Plot 4–6: perdite (solo scaled)
    if mode == "scaled":
        loss_cols = sorted([c for c in res_all.columns
                            if c.startswith("Loss_") and c not in ("Loss_Total","LossPct_Total")])
        if loss_cols:
            for c in loss_cols:
                plt.figure()
                for label, df_g in res_all.groupby("config"):
                    plt.plot(df_g["lambda"], df_g[c], marker="o", label=label)
                plt.xlabel("λ (arrivi/tempo)")
                plt.ylabel(f"Job persi (conteggio) – {c.replace('Loss_', '')}")
                plt.title(f"Job persi {c.replace('Loss_', '')} vs λ")
                plt.grid(True)
                plt.legend()
                plt.show()

        # Perdite totali
        plt.figure()
        for label, df_g in res_all.groupby("config"):
            plt.plot(df_g["lambda"], df_g["Loss_Total"], marker="o", label=label)
        plt.xlabel("λ (arrivi/tempo)")
        plt.ylabel("Perdite totali (conteggio)")
        plt.title("Perdite totali vs λ")
        plt.grid(True)
        plt.legend()
        plt.show()

        # % Perdite totali
        plt.figure()
        for label, df_g in res_all.groupby("config"):
            plt.plot(df_g["lambda"], df_g["LossPct_Total"], marker="o", label=label)
        plt.xlabel("λ (arrivi/tempo)")
        plt.ylabel("Perdite totali (%)")
        plt.title("Percentuale di perdite vs λ")
        plt.grid(True)
        plt.legend()


In [ ]:

res_fast = sweep_and_plot_ABAPA(
    mode="scaled",
    lmb_start=0.5, lmb_end=1.2, lmb_step=0.05,
    mu_A1=5.0, mu_B=1.25, mu_A2=2.5, mu_P=2.5, mu_A3=10.0,
    threshold_time=100000
)


print(res_fast.head())


